In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [ ]:
# -*- coding: utf-8 -*-
"""Untitled1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1kDOZ5aDR611NqiXttVHG_u20bMsbmTqy
"""

# -*- coding: utf-8 -*-
"""
food_waste_pipeline.py

Leakage-free rewrite of the HARE (base linear model + asymmetric-weighted
LightGBM "ghost" residual layer) food waste prediction pipeline.

KEY DIFFERENCES FROM THE ORIGINAL NOTEBOOK
-------------------------------------------
1. THREE-WAY SPLIT (train / val / test), not two-way.
   - train: fit models
   - val:   compare model variants, pick hyperparameters, pick the
            asymmetric penalty. This replaces every place the original
            notebook re-checked `test_data` after a tweak.
   - test:  touched exactly ONCE, at the very end, to report final numbers.
2. Outlier trimming is fit on TRAIN ONLY (quartiles computed from train),
   then the same bounds are applied to val/test. The original computed
   quartiles on the full dataset before splitting, which lets the
   test set influence which training rows get kept.
3. Only ONE model is carried into the final test evaluation. Model
   selection (which ghost model / feature set / penalty) happens on
   validation and is decided before test is ever touched.
4. Cross-validated performance is reported for the final model in
   addition to the single test-set snapshot, since one split's numbers
   aren't strong evidence for a paper on their own.

Run this the same way you'd run the notebook: put train.csv next to it.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, make_scorer,
)
from scipy.stats import uniform, randint
import lightgbm as lgb
import shap

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "python_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5      # kg under-prediction considered "critical"
KG_PER_MEAL = 0.4            # used to convert kg -> "meals saved" narrative


# ============================================================
# 1. LOAD + BASIC CLEANING (safe to do before splitting: none of
#    this uses target-derived statistics or cross-row aggregates)
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )

    numeric_cols = ['meals_served', 'kitchen_staff', 'temperature_C',
                     'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in numeric_cols:
        data[col] = clean_numeric(data[col])

    data['staff_experience'] = (
        data['staff_experience'].astype(str).str.lower().str.strip()
        .replace('nan', np.nan).fillna('unknown')
    )
    data['waste_category'] = data['waste_category'].astype(str).str.lower().str.strip()

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month

    data.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
    # NOTE: 'date' is kept here (only 'ID' dropped) so a chronological split is
    # possible. 'date' is dropped later, after splitting, right before modeling.
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)


# ============================================================
# 2. SPLIT FIRST -- everything downstream is fit on train only
#
# This is a forecasting problem (predict tomorrow's waste from today's
# conditions), so a RANDOM split lets future days leak into training folds
# that get evaluated against past days -- performance looks better than it
# would in real deployment, where you only ever have the past. We split
# chronologically instead: train on the earliest days, validate on the
# next chunk, test on the most recent chunk. Random split is kept as an
# option for comparison, but chronological is the default and what you
# should report as your primary result.
# ============================================================
def three_way_split(data, val_size=0.2, test_size=0.2, chronological=True):
    if chronological:
        data = data.sort_values('date').reset_index(drop=True)
        n = len(data)
        n_test = int(n * test_size)
        n_val = int(n * val_size)
        n_train = n - n_val - n_test
        train = data.iloc[:n_train]
        val = data.iloc[n_train:n_train + n_val]
        test = data.iloc[n_train + n_val:]
        return (train.reset_index(drop=True),
                val.reset_index(drop=True),
                test.reset_index(drop=True))

    train_val, test = train_test_split(data, test_size=test_size, random_state=SEED)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, random_state=SEED)
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


# ============================================================
# 3. OUTLIER TRIMMING -- bounds learned from TRAIN, applied to all splits
# ============================================================
def fit_outlier_bounds(train, cols):
    bounds = {}
    for col in cols:
        Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return bounds


def apply_outlier_bounds(df, bounds):
    mask = pd.Series(True, index=df.index)
    for col, (lo, hi) in bounds.items():
        mask &= df[col].between(lo, hi)
    return df[mask].reset_index(drop=True)


# ============================================================
# 4. FEATURE ENGINEERING -- pure row-wise transforms, safe to reuse
#    across splits since nothing is fit here (no leakage risk)
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg",
    "special_event", "temperature_C", "humidity_percent",
    "day_of_week", "staff_efficiency", "demand_pressure",
    "env_score", "event_risk", "waste_ratio",
    "is_weekend", "day", "month",
]

GHOST_EXTRA_FEATURES = [
    'meals_x_humidity', 'past_waste_x_temp', 'kitchen_x_temp',
    'day_x_humidity', 'meals_served_sq', 'past_waste_kg_sq', 'temperature_C_sq',
]
GHOST_FEATURES = BASE_FEATURES + GHOST_EXTRA_FEATURES


def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / df['kitchen_staff']
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['waste_ratio'] = df['past_waste_kg'] / (df['past_waste_kg'] + 1)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['kitchen_x_temp'] = df['kitchen_staff'] * df['temperature_C']
    df['day_x_humidity'] = df['day'] * df['humidity_percent']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['past_waste_kg_sq'] = df['past_waste_kg'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2
    return df


# ============================================================
# 5. MODELS
# ============================================================
def fit_baseline_lm(train):
    lm = LinearRegression().fit(train[BASE_FEATURES], train['food_waste_kg'])
    return lm


def fit_random_forest(train):
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf.fit(train[BASE_FEATURES], train['food_waste_kg'])
    return rf


def shortfall_mass_scorer(y_true, y_pred):
    """Custom scorer: total kg of dangerous under-prediction. Lower is better,
    so RandomizedSearchCV needs the negative of this."""
    resid = y_true - y_pred
    return np.sum(resid[resid > 0])


critical_shortfall_neg_scorer = make_scorer(
    lambda y_true, y_pred: -shortfall_mass_scorer(y_true, y_pred),
    greater_is_better=True,
)


def fit_hare(train, features, penalty, lgbm_params=None):
    """Base linear learner + asymmetric-weighted LightGBM ghost layer
    that learns the base learner's residuals."""
    base = LinearRegression().fit(train[features], train['food_waste_kg'])
    resid = train['food_waste_kg'].values - base.predict(train[features])
    weights = np.where(resid > 0, penalty, 1.0)

    params = dict(n_estimators=300, learning_rate=0.03, random_state=SEED, verbose=-1)
    if lgbm_params:
        params.update(lgbm_params)  # tuned values (incl. n_estimators/learning_rate) now kept

    ghost = lgb.LGBMRegressor(**params)
    ghost.fit(train[features], resid, sample_weight=weights)

    def predict(df):
        return base.predict(df[features]) + ghost.predict(df[features])

    return base, ghost, predict


from sklearn.base import BaseEstimator, RegressorMixin, clone


class HAREEstimator(BaseEstimator, RegressorMixin):
    """Wraps 'fit linear base -> compute residual -> fit weighted ghost on
    residual -> predict base+ghost' as a SINGLE estimator with the real
    target (food_waste_kg), not a pre-computed residual.

    This matters for cross-validated hyperparameter search: if you instead
    precompute residuals once on the full training set and hand them to
    RandomizedSearchCV, the base model has already seen every row -- including
    the rows that later become each fold's held-out validation slice -- before
    tuning even starts. Wrapping it like this forces the base model to be
    refit on ONLY each fold's training rows, so no fold sees its own
    validation data at any stage.
    """

    def __init__(self, penalty=20.0, n_estimators=300, learning_rate=0.03,
                 num_leaves=31, max_depth=-1, min_child_samples=20,
                 subsample=1.0, colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=0.0):
        self.penalty = penalty
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        self.base_ = LinearRegression().fit(X, y)
        resid = y - self.base_.predict(X)
        weights = np.where(resid > 0, self.penalty, 1.0)
        self.ghost_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, max_depth=self.max_depth,
            min_child_samples=self.min_child_samples, subsample=self.subsample,
            colsample_bytree=self.colsample_bytree, reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda, random_state=SEED, verbose=-1,
        )
        self.ghost_.fit(X, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        return self.base_.predict(X) + self.ghost_.predict(X)


def tune_ghost_hyperparams(train, features, penalty, n_iter=60):
    """Hyperparameter search where base+ghost are refit fresh inside every
    CV fold (see HAREEstimator), scored against the real target so the
    asymmetric penalty's effect on shortfalls is part of what's being tuned."""
    param_dist = {
        'n_estimators': randint(100, 1000),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 60),
        'max_depth': randint(5, 15),
        'min_child_samples': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0, 0.5),
    }
    search = RandomizedSearchCV(
        estimator=HAREEstimator(penalty=penalty),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=critical_shortfall_neg_scorer,  # tune toward low shortfall mass, not just RMSE
        cv=5, random_state=SEED, n_jobs=-1,
    )
    search.fit(train[features], train['food_waste_kg'])
    best = {k: v for k, v in search.best_params_.items()}
    return best


# ============================================================
# 6. METRICS
# ============================================================
def regression_metrics(actual, pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
    }


def shortfall_metrics(actual, pred, threshold=SHORTFALL_THRESHOLD):
    resid = actual - pred
    return {
        'critical_shortfalls': int(np.sum(resid > threshold)),
        'shortfall_mass_kg': float(np.sum(resid[resid > 0])),
        'max_shortfall_kg': float(np.max(resid)),
        'bias_kg': float(np.mean(resid)),
    }


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    bounds = fit_outlier_bounds(train, outlier_cols)
    train = apply_outlier_bounds(train, bounds)
    val = apply_outlier_bounds(val, bounds)
    test = apply_outlier_bounds(test, bounds)
    print(f"After outlier trimming -> Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    # ---------- Baselines ----------
    lm = fit_baseline_lm(train)
    rf = fit_random_forest(train)

    val_pred_lm = lm.predict(val[BASE_FEATURES])
    val_pred_rf = rf.predict(val[BASE_FEATURES])
    print("\n[VAL] Baseline LM:", regression_metrics(val['food_waste_kg'], val_pred_lm))
    print("[VAL] Random Forest:", regression_metrics(val['food_waste_kg'], val_pred_rf))

    # ---------- Model selection on VALIDATION only ----------
    # 7a. Penalty sweep, extended out to w=50 to find where the trade-off
    #     plateaus (this replaces the notebook's test-set penalty search).
    #     We also track RMSE at each penalty so the cost of shortfall
    #     reduction, in accuracy terms, is visible -- this is the
    #     cost-benefit curve for the paper.
    penalty_grid = [1, 2, 3, 5, 7, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50]
    penalty_results = []
    for p in penalty_grid:
        _, _, predict_fn = fit_hare(train, BASE_FEATURES, penalty=p)
        pred = predict_fn(val)
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        r = regression_metrics(val['food_waste_kg'].values, pred)
        penalty_results.append({'penalty': p, **m, 'RMSE': r['RMSE'], 'R2': r['R2']})
    penalty_df = pd.DataFrame(penalty_results)

    # Diminishing-returns selection instead of blind argmin: pick the smallest
    # penalty whose shortfall mass is within 5% of the best mass achieved
    # anywhere in the grid. Blind argmin would always land on w=50 even when
    # w=15 gets 95% of the benefit for a fraction of the accuracy cost.
    best_mass = penalty_df['shortfall_mass_kg'].min()
    within_5pct = penalty_df[penalty_df['shortfall_mass_kg'] <= best_mass * 1.05]
    best_penalty = int(within_5pct['penalty'].min())
    print(f"\n[VAL] Penalty sweep (w = 1..50):")
    print(penalty_df.to_string(index=False))
    print(f"\n[VAL] Chosen penalty (smallest w within 5% of best shortfall mass): {best_penalty}")

    # Cost-benefit plot: shortfall mass (left axis) vs RMSE (right axis) across w
    fig, ax1 = plt.subplots(figsize=(9, 6))
    ax1.plot(penalty_df['penalty'], penalty_df['shortfall_mass_kg'], 'o-', color='darkred', label='Shortfall mass (kg)')
    ax1.set_xlabel('Under-prediction penalty multiplier (w)')
    ax1.set_ylabel('Shortfall mass (kg) — lower is safer', color='darkred')
    ax1.tick_params(axis='y', labelcolor='darkred')
    ax1.axvline(best_penalty, linestyle='--', color='gray', alpha=0.7)
    ax1.text(best_penalty + 0.5, ax1.get_ylim()[1] * 0.9, f'Chosen w={best_penalty}', color='gray')

    ax2 = ax1.twinx()
    ax2.plot(penalty_df['penalty'], penalty_df['RMSE'], 's--', color='steelblue', label='RMSE')
    ax2.set_ylabel('RMSE (kg) — lower is more accurate', color='steelblue')
    ax2.tick_params(axis='y', labelcolor='steelblue')

    plt.title("Operational Safety vs. Accuracy Trade-off\nShortfall risk (red) falls as accuracy (blue) worsens with increasing penalty")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "penalty_cost_benefit_curve.png"), dpi=120)
    plt.close()
    penalty_df.to_csv(os.path.join(OUTPUT_DIR, "penalty_sweep_results.csv"), index=False)

    # 7b. Hyperparameter tuning (train-internal CV, no leakage) at the chosen penalty
    best_params = tune_ghost_hyperparams(train, BASE_FEATURES, best_penalty)
    print(f"\nBest LightGBM ghost params: {best_params}")

    # 7c. Compare feature sets on VALIDATION: base features vs. extended ghost features
    _, _, predict_base_feats = fit_hare(train, BASE_FEATURES, best_penalty, best_params)
    train_fe = engineer_features(train)  # already engineered; kept for clarity
    _, _, predict_ghost_feats = fit_hare(train_fe, GHOST_FEATURES, best_penalty, best_params)

    val_metrics_basefeat = shortfall_metrics(val['food_waste_kg'].values, predict_base_feats(val))
    val_metrics_ghostfeat = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats(val))
    print("\n[VAL] HARE (base features):", val_metrics_basefeat)
    print("[VAL] HARE (extended ghost features):", val_metrics_ghostfeat)

    use_extended = val_metrics_ghostfeat['shortfall_mass_kg'] < val_metrics_basefeat['shortfall_mass_kg']
    final_features = GHOST_FEATURES if use_extended else BASE_FEATURES
    print(f"\n>>> Selected feature set: {'extended' if use_extended else 'base'}")

    # ---------- FINAL MODEL: refit on train, touch test exactly once ----------
    final_base, final_ghost, final_predict = fit_hare(train, final_features, best_penalty, best_params)

    test_pred_hare = final_predict(test)
    test_pred_lm = lm.predict(test[BASE_FEATURES])
    test_pred_rf = rf.predict(test[BASE_FEATURES])
    actual = test['food_waste_kg'].values

    print("\n================ FINAL TEST-SET RESULTS (touched once) ================")
    print("Baseline LM:", regression_metrics(actual, test_pred_lm), shortfall_metrics(actual, test_pred_lm))
    print("Random Forest:", regression_metrics(actual, test_pred_rf), shortfall_metrics(actual, test_pred_rf))
    print("HARE (final):", regression_metrics(actual, test_pred_hare), shortfall_metrics(actual, test_pred_hare))

    # ---------- Cross-validated evidence for the final model (for the paper) ----------
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmse, cv_shortfall_mass = [], []
    # Rolling-origin CV: each fold trains on an earlier chronological block and
    # evaluates on the block right after it -- matches real deployment, where
    # you only ever have the past to predict the future. Do NOT shuffle here.
    full_train_val = pd.concat([train, val], ignore_index=True).sort_values('date').reset_index(drop=True)
    for tr_idx, te_idx in tscv.split(full_train_val):
        fold_train = full_train_val.iloc[tr_idx]
        fold_eval = full_train_val.iloc[te_idx]
        _, _, fold_predict = fit_hare(fold_train, final_features, best_penalty, best_params)
        fold_pred = fold_predict(fold_eval)
        cv_rmse.append(np.sqrt(mean_squared_error(fold_eval['food_waste_kg'], fold_pred)))
        cv_shortfall_mass.append(shortfall_mass_scorer(fold_eval['food_waste_kg'].values, fold_pred))

    print(f"\n5-fold CV (train+val, final model): "
          f"RMSE {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}, "
          f"shortfall mass {np.mean(cv_shortfall_mass):.2f} ± {np.std(cv_shortfall_mass):.2f} kg")

    # ---------- SHAP on final ghost model ----------
    explainer = shap.TreeExplainer(final_ghost)
    shap_sample = test[final_features].iloc[:60]
    shap_values = explainer.shap_values(shap_sample)
    shap_imp = pd.DataFrame({
        'Feature': final_features,
        'Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=shap_imp, x='Importance', y='Feature', hue='Feature',
                palette='crest', legend=False)
    plt.title("Final Model — Ghost Layer Feature Attribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_shap.png"), dpi=120)
    plt.close()

    # ---------- Save final comparison table ----------
    final_table = pd.DataFrame([
        {'Model': 'Baseline LM', **regression_metrics(actual, test_pred_lm), **shortfall_metrics(actual, test_pred_lm)},
        {'Model': 'Random Forest', **regression_metrics(actual, test_pred_rf), **shortfall_metrics(actual, test_pred_rf)},
        {'Model': 'HARE (final)', **regression_metrics(actual, test_pred_hare), **shortfall_metrics(actual, test_pred_hare)},
    ])
    final_table.to_csv(os.path.join(OUTPUT_DIR, "final_test_results.csv"), index=False)
    print(f"\nSaved final results to {OUTPUT_DIR}/final_test_results.csv")
    print(final_table.to_string(index=False))


if __name__ == "__main__":
    main()

# -*- coding: utf-8 -*-
"""
food_waste_pipeline.py

Leakage-free rewrite of the HARE (base linear model + asymmetric-weighted
LightGBM "ghost" residual layer) food waste prediction pipeline.

KEY DIFFERENCES FROM THE ORIGINAL NOTEBOOK
-------------------------------------------
1. THREE-WAY SPLIT (train / val / test), not two-way.
   - train: fit models
   - val:   compare model variants, pick hyperparameters, pick the
            asymmetric penalty. This replaces every place the original
            notebook re-checked `test_data` after a tweak.
   - test:  touched exactly ONCE, at the very end, to report final numbers.
2. Outlier trimming is fit on TRAIN ONLY (quartiles computed from train),
   then the same bounds are applied to val/test. The original computed
   quartiles on the full dataset before splitting, which lets the
   test set influence which training rows get kept.
3. Only ONE model is carried into the final test evaluation. Model
   selection (which ghost model / feature set / penalty) happens on
   validation and is decided before test is ever touched.
4. Cross-validated performance is reported for the final model in
   addition to the single test-set snapshot, since one split's numbers
   aren't strong evidence for a paper on their own.

Run this the same way you'd run the notebook: put train.csv next to it.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, make_scorer,
)
from scipy.stats import uniform, randint
import lightgbm as lgb
import shap

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "python_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5      # kg under-prediction considered "critical"
KG_PER_MEAL = 0.4            # used to convert kg -> "meals saved" narrative


# ============================================================
# 1. LOAD + BASIC CLEANING (safe to do before splitting: none of
#    this uses target-derived statistics or cross-row aggregates)
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )

    numeric_cols = ['meals_served', 'kitchen_staff', 'temperature_C',
                     'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in numeric_cols:
        data[col] = clean_numeric(data[col])

    data['staff_experience'] = (
        data['staff_experience'].astype(str).str.lower().str.strip()
        .replace('nan', np.nan).fillna('unknown')
    )
    data['waste_category'] = data['waste_category'].astype(str).str.lower().str.strip()

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month

    data.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
    # NOTE: 'date' is kept here (only 'ID' dropped) so a chronological split is
    # possible. 'date' is dropped later, after splitting, right before modeling.
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)


# ============================================================
# 2. SPLIT FIRST -- everything downstream is fit on train only
#
# This is a forecasting problem (predict tomorrow's waste from today's
# conditions), so a RANDOM split lets future days leak into training folds
# that get evaluated against past days -- performance looks better than it
# would in real deployment, where you only ever have the past. We split
# chronologically instead: train on the earliest days, validate on the
# next chunk, test on the most recent chunk. Random split is kept as an
# option for comparison, but chronological is the default and what you
# should report as your primary result.
# ============================================================
def three_way_split(data, val_size=0.2, test_size=0.2, chronological=True):
    if chronological:
        data = data.sort_values('date').reset_index(drop=True)
        n = len(data)
        n_test = int(n * test_size)
        n_val = int(n * val_size)
        n_train = n - n_val - n_test
        train = data.iloc[:n_train]
        val = data.iloc[n_train:n_train + n_val]
        test = data.iloc[n_train + n_val:]
        return (train.reset_index(drop=True),
                val.reset_index(drop=True),
                test.reset_index(drop=True))

    train_val, test = train_test_split(data, test_size=test_size, random_state=SEED)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, random_state=SEED)
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


# ============================================================
# 3. OUTLIER TRIMMING -- bounds learned from TRAIN, applied to all splits
# ============================================================
def fit_outlier_bounds(train, cols):
    bounds = {}
    for col in cols:
        Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return bounds


def apply_outlier_bounds(df, bounds):
    mask = pd.Series(True, index=df.index)
    for col, (lo, hi) in bounds.items():
        mask &= df[col].between(lo, hi)
    return df[mask].reset_index(drop=True)


# ============================================================
# 4. FEATURE ENGINEERING -- pure row-wise transforms, safe to reuse
#    across splits since nothing is fit here (no leakage risk)
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg",
    "special_event", "temperature_C", "humidity_percent",
    "day_of_week", "staff_efficiency", "demand_pressure",
    "env_score", "event_risk", "waste_ratio",
    "is_weekend", "day", "month",
]

GHOST_EXTRA_FEATURES = [
    'meals_x_humidity', 'past_waste_x_temp', 'kitchen_x_temp',
    'day_x_humidity', 'meals_served_sq', 'past_waste_kg_sq', 'temperature_C_sq',
]
GHOST_FEATURES = BASE_FEATURES + GHOST_EXTRA_FEATURES


def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / df['kitchen_staff']
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['waste_ratio'] = df['past_waste_kg'] / (df['past_waste_kg'] + 1)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['kitchen_x_temp'] = df['kitchen_staff'] * df['temperature_C']
    df['day_x_humidity'] = df['day'] * df['humidity_percent']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['past_waste_kg_sq'] = df['past_waste_kg'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2
    return df


# ============================================================
# 5. MODELS
# ============================================================
def fit_baseline_lm(train):
    lm = LinearRegression().fit(train[BASE_FEATURES], train['food_waste_kg'])
    return lm


def fit_random_forest(train):
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf.fit(train[BASE_FEATURES], train['food_waste_kg'])
    return rf


def shortfall_mass_scorer(y_true, y_pred):
    """Custom scorer: total kg of dangerous under-prediction. Lower is better,
    so RandomizedSearchCV needs the negative of this."""
    resid = y_true - y_pred
    return np.sum(resid[resid > 0])


critical_shortfall_neg_scorer = make_scorer(
    lambda y_true, y_pred: -shortfall_mass_scorer(y_true, y_pred),
    greater_is_better=True,
)


def fit_hare(train, features, penalty, lgbm_params=None):
    """Base linear learner + asymmetric-weighted LightGBM ghost layer
    that learns the base learner's residuals."""
    base = LinearRegression().fit(train[features], train['food_waste_kg'])
    resid = train['food_waste_kg'].values - base.predict(train[features])
    weights = np.where(resid > 0, penalty, 1.0)

    params = dict(n_estimators=300, learning_rate=0.03, random_state=SEED, verbose=-1)
    if lgbm_params:
        params.update(lgbm_params)  # tuned values (incl. n_estimators/learning_rate) now kept

    ghost = lgb.LGBMRegressor(**params)
    ghost.fit(train[features], resid, sample_weight=weights)

    def predict(df):
        return base.predict(df[features]) + ghost.predict(df[features])

    return base, ghost, predict


from sklearn.base import BaseEstimator, RegressorMixin, clone


class HAREEstimator(BaseEstimator, RegressorMixin):
    """Wraps 'fit linear base -> compute residual -> fit weighted ghost on
    residual -> predict base+ghost' as a SINGLE estimator with the real
    target (food_waste_kg), not a pre-computed residual.

    This matters for cross-validated hyperparameter search: if you instead
    precompute residuals once on the full training set and hand them to
    RandomizedSearchCV, the base model has already seen every row -- including
    the rows that later become each fold's held-out validation slice -- before
    tuning even starts. Wrapping it like this forces the base model to be
    refit on ONLY each fold's training rows, so no fold sees its own
    validation data at any stage.
    """

    def __init__(self, penalty=20.0, n_estimators=300, learning_rate=0.03,
                 num_leaves=31, max_depth=-1, min_child_samples=20,
                 subsample=1.0, colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=0.0):
        self.penalty = penalty
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        self.base_ = LinearRegression().fit(X, y)
        resid = y - self.base_.predict(X)
        weights = np.where(resid > 0, self.penalty, 1.0)
        self.ghost_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, max_depth=self.max_depth,
            min_child_samples=self.min_child_samples, subsample=self.subsample,
            colsample_bytree=self.colsample_bytree, reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda, random_state=SEED, verbose=-1,
        )
        self.ghost_.fit(X, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        return self.base_.predict(X) + self.ghost_.predict(X)


def tune_ghost_hyperparams(train, features, penalty, n_iter=60):
    """Hyperparameter search where base+ghost are refit fresh inside every
    CV fold (see HAREEstimator), scored against the real target so the
    asymmetric penalty's effect on shortfalls is part of what's being tuned."""
    param_dist = {
        'n_estimators': randint(100, 1000),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 60),
        'max_depth': randint(5, 15),
        'min_child_samples': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0, 0.5),
    }
    search = RandomizedSearchCV(
        estimator=HAREEstimator(penalty=penalty),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=critical_shortfall_neg_scorer,  # tune toward low shortfall mass, not just RMSE
        cv=5, random_state=SEED, n_jobs=-1,
    )
    search.fit(train[features], train['food_waste_kg'])
    best = {k: v for k, v in search.best_params_.items()}
    return best


# ============================================================
# 6. METRICS
# ============================================================
def regression_metrics(actual, pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
    }


def shortfall_metrics(actual, pred, threshold=SHORTFALL_THRESHOLD):
    resid = actual - pred
    return {
        'critical_shortfalls': int(np.sum(resid > threshold)),
        'shortfall_mass_kg': float(np.sum(resid[resid > 0])),
        'max_shortfall_kg': float(np.max(resid)),
        'bias_kg': float(np.mean(resid)),
    }


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    bounds = fit_outlier_bounds(train, outlier_cols)
    train = apply_outlier_bounds(train, bounds)
    val = apply_outlier_bounds(val, bounds)
    test = apply_outlier_bounds(test, bounds)
    print(f"After outlier trimming -> Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    # ---------- Baselines ----------
    lm = fit_baseline_lm(train)
    rf = fit_random_forest(train)

    val_pred_lm = lm.predict(val[BASE_FEATURES])
    val_pred_rf = rf.predict(val[BASE_FEATURES])
    print("\n[VAL] Baseline LM:", regression_metrics(val['food_waste_kg'], val_pred_lm))
    print("[VAL] Random Forest:", regression_metrics(val['food_waste_kg'], val_pred_rf))

    # ---------- Model selection on VALIDATION only ----------
    # 7a. Penalty sweep (this replaces the notebook's test-set penalty search)
    penalty_grid = [1, 2, 3, 5, 7, 10, 12, 15, 20]
    penalty_results = []
    for p in penalty_grid:
        _, _, predict_fn = fit_hare(train, BASE_FEATURES, penalty=p)
        m = shortfall_metrics(val['food_waste_kg'].values, predict_fn(val))
        penalty_results.append({'penalty': p, **m})
    penalty_df = pd.DataFrame(penalty_results)
    best_penalty = int(penalty_df.loc[penalty_df['shortfall_mass_kg'].idxmin(), 'penalty'])
    print(f"\n[VAL] Chosen penalty (min shortfall mass on val): {best_penalty}")
    print(penalty_df.to_string(index=False))

    # 7b. Hyperparameter tuning (train-internal CV, no leakage) at the chosen penalty
    best_params = tune_ghost_hyperparams(train, BASE_FEATURES, best_penalty)
    print(f"\nBest LightGBM ghost params: {best_params}")

    # 7c. Compare feature sets on VALIDATION: base features vs. extended ghost features
    _, _, predict_base_feats = fit_hare(train, BASE_FEATURES, best_penalty, best_params)
    train_fe = engineer_features(train)  # already engineered; kept for clarity
    _, _, predict_ghost_feats = fit_hare(train_fe, GHOST_FEATURES, best_penalty, best_params)

    val_metrics_basefeat = shortfall_metrics(val['food_waste_kg'].values, predict_base_feats(val))
    val_metrics_ghostfeat = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats(val))
    print("\n[VAL] HARE (base features):", val_metrics_basefeat)
    print("[VAL] HARE (extended ghost features):", val_metrics_ghostfeat)

    use_extended = val_metrics_ghostfeat['shortfall_mass_kg'] < val_metrics_basefeat['shortfall_mass_kg']
    final_features = GHOST_FEATURES if use_extended else BASE_FEATURES
    print(f"\n>>> Selected feature set: {'extended' if use_extended else 'base'}")

    # ---------- FINAL MODEL: refit on train, touch test exactly once ----------
    final_base, final_ghost, final_predict = fit_hare(train, final_features, best_penalty, best_params)

    test_pred_hare = final_predict(test)
    test_pred_lm = lm.predict(test[BASE_FEATURES])
    test_pred_rf = rf.predict(test[BASE_FEATURES])
    actual = test['food_waste_kg'].values

    print("\n================ FINAL TEST-SET RESULTS (touched once) ================")
    print("Baseline LM:", regression_metrics(actual, test_pred_lm), shortfall_metrics(actual, test_pred_lm))
    print("Random Forest:", regression_metrics(actual, test_pred_rf), shortfall_metrics(actual, test_pred_rf))
    print("HARE (final):", regression_metrics(actual, test_pred_hare), shortfall_metrics(actual, test_pred_hare))

    # ---------- Cross-validated evidence for the final model (for the paper) ----------
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmse, cv_shortfall_mass = [], []
    # Rolling-origin CV: each fold trains on an earlier chronological block and
    # evaluates on the block right after it -- matches real deployment, where
    # you only ever have the past to predict the future. Do NOT shuffle here.
    full_train_val = pd.concat([train, val], ignore_index=True).sort_values('date').reset_index(drop=True)
    for tr_idx, te_idx in tscv.split(full_train_val):
        fold_train = full_train_val.iloc[tr_idx]
        fold_eval = full_train_val.iloc[te_idx]
        _, _, fold_predict = fit_hare(fold_train, final_features, best_penalty, best_params)
        fold_pred = fold_predict(fold_eval)
        cv_rmse.append(np.sqrt(mean_squared_error(fold_eval['food_waste_kg'], fold_pred)))
        cv_shortfall_mass.append(shortfall_mass_scorer(fold_eval['food_waste_kg'].values, fold_pred))

    print(f"\n5-fold CV (train+val, final model): "
          f"RMSE {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}, "
          f"shortfall mass {np.mean(cv_shortfall_mass):.2f} ± {np.std(cv_shortfall_mass):.2f} kg")

    # ---------- SHAP on final ghost model ----------
    explainer = shap.TreeExplainer(final_ghost)
    shap_sample = test[final_features].iloc[:60]
    shap_values = explainer.shap_values(shap_sample)
    shap_imp = pd.DataFrame({
        'Feature': final_features,
        'Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=shap_imp, x='Importance', y='Feature', hue='Feature',
                palette='crest', legend=False)
    plt.title("Final Model — Ghost Layer Feature Attribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_shap.png"), dpi=120)
    plt.close()

    # ---------- Save final comparison table ----------
    final_table = pd.DataFrame([
        {'Model': 'Baseline LM', **regression_metrics(actual, test_pred_lm), **shortfall_metrics(actual, test_pred_lm)},
        {'Model': 'Random Forest', **regression_metrics(actual, test_pred_rf), **shortfall_metrics(actual, test_pred_rf)},
        {'Model': 'HARE (final)', **regression_metrics(actual, test_pred_hare), **shortfall_metrics(actual, test_pred_hare)},
    ])
    final_table.to_csv(os.path.join(OUTPUT_DIR, "final_test_results.csv"), index=False)
    print(f"\nSaved final results to {OUTPUT_DIR}/final_test_results.csv")
    print(final_table.to_string(index=False))


if __name__ == "__main__":
    main()

# -*- coding: utf-8 -*-
"""
food_waste_pipeline.py

Leakage-free rewrite of the HARE (base linear model + asymmetric-weighted
LightGBM "ghost" residual layer) food waste prediction pipeline.

KEY DIFFERENCES FROM THE ORIGINAL NOTEBOOK
-------------------------------------------
1. THREE-WAY SPLIT (train / val / test), not two-way.
   - train: fit models
   - val:   compare model variants, pick hyperparameters, pick the
            asymmetric penalty. This replaces every place the original
            notebook re-checked `test_data` after a tweak.
   - test:  touched exactly ONCE, at the very end, to report final numbers.
2. Outlier trimming is fit on TRAIN ONLY (quartiles computed from train),
   then the same bounds are applied to val/test. The original computed
   quartiles on the full dataset before splitting, which lets the
   test set influence which training rows get kept.
3. Only ONE model is carried into the final test evaluation. Model
   selection (which ghost model / feature set / penalty) happens on
   validation and is decided before test is ever touched.
4. Cross-validated performance is reported for the final model in
   addition to the single test-set snapshot, since one split's numbers
   aren't strong evidence for a paper on their own.

Run this the same way you'd run the notebook: put train.csv next to it.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, make_scorer,
)
from scipy.stats import uniform, randint
import lightgbm as lgb
import shap

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "python_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5      # kg under-prediction considered "critical"
KG_PER_MEAL = 0.4            # used to convert kg -> "meals saved" narrative

# ---- OPERATIONAL COST ASSUMPTIONS ------------------------------------
# These turn "kg of shortfall" and "kg of surplus" into a single comparable
# number so we can pick a penalty (and compare models) on cost, not on
# shortfall mass alone or on R2/RMSE alone. The two are NOT the same kind
# of cost: running short means lost sales, refunds, and reputational
# damage; over-preparing just means wasted ingredients. That's WHY the
# asymmetric penalty exists in the first place, but until now nothing
# converted that intuition into one number you could optimize against.
#
# The values below are placeholders (shortfall costs 3x more per kg than
# surplus, a common assumption in newsvendor/inventory literature) --
# replace them with your kitchen's real figures (ingredient cost per kg
# for surplus, and a defensible estimate of lost-sale + reputational cost
# per kg for shortfall) before this goes in the paper. State the
# assumption explicitly in your methods section either way.
COST_PER_KG_SHORTFALL = 3.0
COST_PER_KG_SURPLUS = 1.0


# ============================================================
# 1. LOAD + BASIC CLEANING (safe to do before splitting: none of
#    this uses target-derived statistics or cross-row aggregates)
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )

    numeric_cols = ['meals_served', 'kitchen_staff', 'temperature_C',
                     'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in numeric_cols:
        data[col] = clean_numeric(data[col])

    data['staff_experience'] = (
        data['staff_experience'].astype(str).str.lower().str.strip()
        .replace('nan', np.nan).fillna('unknown')
    )
    data['waste_category'] = data['waste_category'].astype(str).str.lower().str.strip()

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month

    data.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
    # NOTE: 'date' is kept here (only 'ID' dropped) so a chronological split is
    # possible. 'date' is dropped later, after splitting, right before modeling.
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)


# ============================================================
# 2. SPLIT FIRST -- everything downstream is fit on train only
#
# This is a forecasting problem (predict tomorrow's waste from today's
# conditions), so a RANDOM split lets future days leak into training folds
# that get evaluated against past days -- performance looks better than it
# would in real deployment, where you only ever have the past. We split
# chronologically instead: train on the earliest days, validate on the
# next chunk, test on the most recent chunk. Random split is kept as an
# option for comparison, but chronological is the default and what you
# should report as your primary result.
# ============================================================
def three_way_split(data, val_size=0.2, test_size=0.2, chronological=True):
    if chronological:
        data = data.sort_values('date').reset_index(drop=True)
        n = len(data)
        n_test = int(n * test_size)
        n_val = int(n * val_size)
        n_train = n - n_val - n_test
        train = data.iloc[:n_train]
        val = data.iloc[n_train:n_train + n_val]
        test = data.iloc[n_train + n_val:]
        return (train.reset_index(drop=True),
                val.reset_index(drop=True),
                test.reset_index(drop=True))

    train_val, test = train_test_split(data, test_size=test_size, random_state=SEED)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, random_state=SEED)
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


# ============================================================
# 3. OUTLIER TRIMMING -- bounds learned from TRAIN, applied to all splits
# ============================================================
def fit_outlier_bounds(train, cols):
    bounds = {}
    for col in cols:
        Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return bounds


def apply_outlier_bounds(df, bounds):
    mask = pd.Series(True, index=df.index)
    for col, (lo, hi) in bounds.items():
        mask &= df[col].between(lo, hi)
    return df[mask].reset_index(drop=True)


# ============================================================
# 4. FEATURE ENGINEERING -- pure row-wise transforms, safe to reuse
#    across splits since nothing is fit here (no leakage risk)
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg",
    "special_event", "temperature_C", "humidity_percent",
    "day_of_week", "staff_efficiency", "demand_pressure",
    "env_score", "event_risk", "waste_ratio",
    "is_weekend", "day", "month",
]

GHOST_EXTRA_FEATURES = [
    'meals_x_humidity', 'past_waste_x_temp', 'kitchen_x_temp',
    'day_x_humidity', 'meals_served_sq', 'past_waste_kg_sq', 'temperature_C_sq',
]
GHOST_FEATURES = BASE_FEATURES + GHOST_EXTRA_FEATURES

# From the friend's code review: these two are genuinely new (not
# redundant with what's above) and worth testing on validation. Kept as a
# SEPARATE candidate set rather than merged in blind, so the pipeline can
# show on val whether they actually help before they're trusted.
FRIEND_EXTRA_FEATURES = ['temp_humidity_index', 'meal_humidity_ratio']
GHOST_FEATURES_V2 = GHOST_FEATURES + FRIEND_EXTRA_FEATURES


def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / df['kitchen_staff']
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['waste_ratio'] = df['past_waste_kg'] / (df['past_waste_kg'] + 1)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['kitchen_x_temp'] = df['kitchen_staff'] * df['temperature_C']
    df['day_x_humidity'] = df['day'] * df['humidity_percent']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['past_waste_kg_sq'] = df['past_waste_kg'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2

    # Friend's additions (kept separate from the set above; tested, not assumed)
    df['temp_humidity_index'] = df['temperature_C'] + 0.1 * df['humidity_percent']
    df['meal_humidity_ratio'] = df['meals_served'] / (df['humidity_percent'] + 1)
    return df


# ============================================================
# 5. MODELS
# ============================================================
def fit_baseline_lm(train):
    lm = LinearRegression().fit(train[BASE_FEATURES], train['food_waste_kg'])
    return lm


def fit_random_forest(train):
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf.fit(train[BASE_FEATURES], train['food_waste_kg'])
    return rf


def shortfall_mass_scorer(y_true, y_pred):
    """Custom scorer: total kg of dangerous under-prediction. Lower is better,
    so RandomizedSearchCV needs the negative of this."""
    resid = y_true - y_pred
    return np.sum(resid[resid > 0])


critical_shortfall_neg_scorer = make_scorer(
    lambda y_true, y_pred: -shortfall_mass_scorer(y_true, y_pred),
    greater_is_better=True,
)


def _make_ghost_model(ghost_type, params, seed=SEED):
    """Build a ghost-layer regressor. All three backends are trained with
    the SAME asymmetric sample_weight -- the point is to compare which
    tree library learns the weighted residual best, not to drop the
    asymmetric weighting the way the unweighted XGBoost version did.

    DETERMINISM: each backend is pinned to single-threaded, bit-reproducible
    training. Fixing random_state alone is NOT enough for gradient-boosted
    trees -- multi-threaded histogram building sums floating-point values in
    a thread-race-dependent order, which changes split points slightly from
    run to run even with an identical seed and identical data. That's why
    the same script produced RMSE 6.700 / 6.573 / 6.676 on three "identical"
    runs. n_jobs=1 (LightGBM/XGBoost) and thread_count=1 (CatBoost) remove
    that source of variance; LightGBM additionally needs deterministic=True
    and force_row_wise=True, since its default multi-threaded histogram
    path can be nondeterministic even at n_jobs=1 without these flags.
    """
    if ghost_type == 'lightgbm':
        base_params = dict(n_estimators=300, learning_rate=0.03, random_state=seed, verbose=-1,
                            n_jobs=1, deterministic=True, force_row_wise=True)
        base_params.update(params or {})
        return lgb.LGBMRegressor(**base_params)
    elif ghost_type == 'xgboost':
        import xgboost as xgb
        base_params = dict(n_estimators=300, learning_rate=0.03, random_state=seed,
                            objective='reg:squarederror', n_jobs=1)
        base_params.update(params or {})
        return xgb.XGBRegressor(**base_params)
    elif ghost_type == 'catboost':
        import catboost as cb
        base_params = dict(iterations=300, learning_rate=0.03, random_seed=seed, verbose=False,
                            thread_count=1)
        base_params.update(params or {})
        return cb.CatBoostRegressor(**base_params)
    else:
        raise ValueError(f"Unknown ghost_type: {ghost_type}")


def fit_hare(train, features, penalty, lgbm_params=None, ghost_type='lightgbm'):
    """Base linear learner + asymmetric-weighted ghost layer (LightGBM by
    default; pass ghost_type='xgboost' or 'catboost' to swap backends)
    that learns the base learner's residuals."""
    base = LinearRegression().fit(train[features], train['food_waste_kg'])
    resid = train['food_waste_kg'].values - base.predict(train[features])
    weights = np.where(resid > 0, penalty, 1.0)

    ghost = _make_ghost_model(ghost_type, lgbm_params)
    ghost.fit(train[features], resid, sample_weight=weights)

    def predict(df):
        return base.predict(df[features]) + ghost.predict(df[features])

    return base, ghost, predict


from sklearn.base import BaseEstimator, RegressorMixin, clone


class HAREEstimator(BaseEstimator, RegressorMixin):
    """Wraps 'fit linear base -> compute residual -> fit weighted ghost on
    residual -> predict base+ghost' as a SINGLE estimator with the real
    target (food_waste_kg), not a pre-computed residual.

    This matters for cross-validated hyperparameter search: if you instead
    precompute residuals once on the full training set and hand them to
    RandomizedSearchCV, the base model has already seen every row -- including
    the rows that later become each fold's held-out validation slice -- before
    tuning even starts. Wrapping it like this forces the base model to be
    refit on ONLY each fold's training rows, so no fold sees its own
    validation data at any stage.
    """

    def __init__(self, penalty=20.0, n_estimators=300, learning_rate=0.03,
                 num_leaves=31, max_depth=-1, min_child_samples=20,
                 subsample=1.0, colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=0.0):
        self.penalty = penalty
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        self.base_ = LinearRegression().fit(X, y)
        resid = y - self.base_.predict(X)
        weights = np.where(resid > 0, self.penalty, 1.0)
        self.ghost_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, max_depth=self.max_depth,
            min_child_samples=self.min_child_samples, subsample=self.subsample,
            colsample_bytree=self.colsample_bytree, reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda, random_state=SEED, verbose=-1,
            # Pinned for determinism -- see _make_ghost_model docstring for why
            # random_state alone doesn't guarantee reproducible LightGBM fits.
            n_jobs=1, deterministic=True, force_row_wise=True,
        )

        self.ghost_.fit(X, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        return self.base_.predict(X) + self.ghost_.predict(X)


def tune_ghost_hyperparams(train, features, penalty, n_iter=60):
    """Hyperparameter search where base+ghost are refit fresh inside every
    CV fold (see HAREEstimator), scored against the real target so the
    asymmetric penalty's effect on shortfalls is part of what's being tuned.

    n_jobs on the search itself is left at -1 (parallelizes across CV folds/
    candidates via joblib, which is safe and deterministic given a fixed
    random_state) -- each individual LGBMRegressor inside HAREEstimator is
    pinned to n_jobs=1 so folds running concurrently don't oversubscribe
    threads or hit LightGBM's nondeterministic multi-threaded path.
    """
    param_dist = {
        'n_estimators': randint(100, 1000),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 60),
        'max_depth': randint(5, 15),
        'min_child_samples': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0, 0.5),
    }
    search = RandomizedSearchCV(
        estimator=HAREEstimator(penalty=penalty),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=critical_shortfall_neg_scorer,  # tune toward low shortfall mass, not just RMSE
        cv=5, random_state=SEED, n_jobs=-1,
    )
    search.fit(train[features], train['food_waste_kg'])
    best = {k: v for k, v in search.best_params_.items()}
    return best


# ============================================================
# 6. METRICS
# ============================================================
def regression_metrics(actual, pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
    }


def shortfall_metrics(actual, pred, threshold=SHORTFALL_THRESHOLD):
    resid = actual - pred  # positive = under-prediction (shortfall), negative = over-prediction (surplus)
    shortfall_mass = float(np.sum(resid[resid > 0]))
    surplus_mass = float(np.sum(-resid[resid < 0]))
    total_cost = shortfall_mass * COST_PER_KG_SHORTFALL + surplus_mass * COST_PER_KG_SURPLUS
    return {
        'critical_shortfalls': int(np.sum(resid > threshold)),
        'shortfall_mass_kg': shortfall_mass,
        'surplus_mass_kg': surplus_mass,
        'total_cost': total_cost,
        'max_shortfall_kg': float(np.max(resid)),
        'bias_kg': float(np.mean(resid)),
    }


# ============================================================
# 6b. COST-RATIO SENSITIVITY ANALYSIS (operational domain)
# ============================================================
# Total operational cost is linear in the shortfall:surplus cost ratio r:
#   cost(r) = shortfall_mass_kg * r + surplus_mass_kg          (surplus cost fixed at 1)
# Two models' cost lines cross at exactly one r (the "breakeven ratio"):
#   r* = (surplus_B - surplus_A) / (shortfall_A - shortfall_B)
# This is exact arithmetic on the test-set masses already computed --
# no retraining, no extra assumptions beyond the linear cost model already
# used everywhere else in this pipeline. It answers "how sensitive is the
# 'HARE is best' claim to the COST_PER_KG_SHORTFALL/SURPLUS assumption?"
# explicitly, instead of reporting a single number at one assumed ratio.
def compute_breakeven_ratio(short_a, surplus_a, short_b, surplus_b):
    """r at which cost_a(r) == cost_b(r). Returns None if the lines are
    parallel (equal shortfall mass -> no finite crossing)."""
    denom = short_a - short_b
    if denom == 0:
        return None
    return (surplus_b - surplus_a) / denom


def cost_ratio_sensitivity(shortfall_masses, surplus_masses, reference_model,
                            r_range=(1.0, 5.0), r_step=0.05):
    """
    Sensitivity analysis of total operational cost to the shortfall:surplus
    cost ratio r, for every model in shortfall_masses/surplus_masses.

    Produces:
      - {OUTPUT_DIR}/cost_ratio_sensitivity.csv   (cost curves, all models, all r)
      - {OUTPUT_DIR}/cost_ratio_sensitivity.png   (the sensitivity plot)
      - printed breakeven ratios: reference_model vs every other model

    This does NOT retrain anything -- it is pure post-hoc arithmetic on
    shortfall_mass_kg / surplus_mass_kg already produced by shortfall_metrics().
    Reports the exact breakeven point rather than a single cost number at
    one assumed ratio, so the "best model" claim is shown with its domain
    of validity, not asserted unconditionally.
    """
    models = list(shortfall_masses.keys())
    r_vals = np.arange(r_range[0], r_range[1] + 1e-9, r_step)

    curve_rows = []
    for r in r_vals:
        row = {'r': round(float(r), 4)}
        for m in models:
            row[m] = shortfall_masses[m] * r + surplus_masses[m]
        curve_rows.append(row)
    curve_df = pd.DataFrame(curve_rows)
    curve_df.to_csv(os.path.join(OUTPUT_DIR, "cost_ratio_sensitivity.csv"), index=False)

    breakevens = {}
    print(f"\n================ COST-RATIO SENSITIVITY (reference: {reference_model}) ================")
    for m in models:
        if m == reference_model:
            continue
        r_star = compute_breakeven_ratio(
            shortfall_masses[reference_model], surplus_masses[reference_model],
            shortfall_masses[m], surplus_masses[m],
        )
        breakevens[m] = r_star
        if r_star is None:
            print(f"{reference_model} vs {m}: shortfall masses equal -> no finite breakeven "
                  f"(one model dominates at every r)")
        else:
            direction = ("wins for r >" if shortfall_masses[reference_model] < shortfall_masses[m]
                         else "wins for r <")
            print(f"{reference_model} vs {m}: breakeven r* = {r_star:.3f}  "
                  f"({reference_model} {direction} {r_star:.3f})")

    plt.figure(figsize=(9, 6))
    for m in models:
        style = '-' if m == reference_model else '--'
        lw = 2.5 if m == reference_model else 1.5
        plt.plot(curve_df['r'], curve_df[m], style, linewidth=lw, label=m)
    ymax = curve_df[models].values.max()
    for m, r_star in breakevens.items():
        if r_star is not None and r_range[0] <= r_star <= r_range[1]:
            plt.axvline(r_star, linestyle=':', color='gray', alpha=0.7)
            plt.text(r_star + 0.03, ymax * 0.95, f'r*={r_star:.2f}\n({m})',
                      fontsize=8, color='gray', va='top')
    plt.xlabel('Shortfall-to-surplus cost ratio (r)')
    plt.ylabel('Total operational cost (test set)')
    plt.title(f'Cost-ratio sensitivity: total operational cost vs r\n'
              f'({reference_model} is the reference / proposed method)')
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "cost_ratio_sensitivity.png"), dpi=120)
    plt.close()

    print(f"\nSaved sensitivity curve data to {OUTPUT_DIR}/cost_ratio_sensitivity.csv")
    print(f"Saved sensitivity plot to {OUTPUT_DIR}/cost_ratio_sensitivity.png")

    return curve_df, breakevens


# ============================================================
# 6c. CONTROL BASELINES: constant safety buffer, quantile regression
# ============================================================
# Both answer the same reviewer question from two different angles:
# "Why does HARE need a two-stage residual model at all?"
#
#   - Safety buffer: the cheapest possible fix -- take the linear model and
#     add a constant kg to every prediction. If HARE just reduces to "the
#     linear model plus a blanket offset", a constant shift should match it.
#     Tuned on val by the SAME total-cost criterion as HARE's penalty w,
#     so the comparison is apples-to-apples.
#   - Quantile regression: the standard tool for asymmetric-loss demand
#     prediction in the OR/inventory literature (newsvendor-style). Trained
#     directly on the pinball loss at a chosen quantile, with NO base model
#     or residual step. If quantile LightGBM matches HARE's cost, the
#     two-stage base+ghost architecture isn't earning its complexity.
def fit_safety_buffer(train, val, features, k_grid=None):
    """Baseline LM + constant offset k, tuned on val to minimize total cost
    (same objective HARE's penalty sweep uses). No per-row adaptivity --
    every prediction gets the same additive buffer regardless of that row's
    predicted risk."""
    if k_grid is None:
        k_grid = np.arange(0, 15.25, 0.25)

    lm = LinearRegression().fit(train[features], train['food_waste_kg'])
    val_base_pred = lm.predict(val[features])

    results = []
    for k in k_grid:
        pred = val_base_pred + k
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        results.append({'k': float(k), **m})
    k_df = pd.DataFrame(results)
    best_k = float(k_df.loc[k_df['total_cost'].idxmin(), 'k'])

    def predict(df):
        return lm.predict(df[features]) + best_k

    return lm, best_k, predict, k_df


def fit_quantile_lgbm(train, features, quantile, lgbm_params=None):
    """Single-stage LightGBM trained directly on pinball loss at the given
    quantile -- no base linear model, no residual stage. The standard
    alternative to HARE's architecture for asymmetric-loss prediction."""
    base_params = dict(
        objective='quantile', alpha=quantile,
        n_estimators=300, learning_rate=0.03, random_state=SEED, verbose=-1,
        n_jobs=1, deterministic=True, force_row_wise=True,
    )
    base_params.update(lgbm_params or {})
    model = lgb.LGBMRegressor(**base_params)
    model.fit(train[features], train['food_waste_kg'])

    def predict(df):
        return model.predict(df[features])

    return model, predict


def tune_quantile_lgbm(train, val, features, quantile_grid=None):
    """Sweep the target quantile on val (analogous to HARE's penalty sweep),
    select by total operational cost, using default LightGBM hyperparams
    for a fair "out of the box" comparison against HARE's tuned ghost."""
    if quantile_grid is None:
        quantile_grid = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

    results = []
    models = {}
    for q in quantile_grid:
        model, predict_fn = fit_quantile_lgbm(train, features, q)
        pred = predict_fn(val)
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        results.append({'quantile': q, **m})
        models[q] = (model, predict_fn)
    q_df = pd.DataFrame(results)
    best_q = float(q_df.loc[q_df['total_cost'].idxmin(), 'quantile'])
    return models[best_q][0], models[best_q][1], best_q, q_df


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    bounds = fit_outlier_bounds(train, outlier_cols)
    train = apply_outlier_bounds(train, bounds)
    val = apply_outlier_bounds(val, bounds)
    test = apply_outlier_bounds(test, bounds)
    print(f"After outlier trimming -> Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    # ---------- Baselines ----------
    lm = fit_baseline_lm(train)
    rf = fit_random_forest(train)

    val_pred_lm = lm.predict(val[BASE_FEATURES])
    val_pred_rf = rf.predict(val[BASE_FEATURES])
    print("\n[VAL] Baseline LM:", regression_metrics(val['food_waste_kg'], val_pred_lm))
    print("[VAL] Random Forest:", regression_metrics(val['food_waste_kg'], val_pred_rf))

    # ---------- Control baselines (chosen on VAL, same as HARE's penalty) ----------
    # These directly answer "why not just do something simpler?" -- selected
    # by the identical total-cost criterion as HARE's penalty w, so every
    # model in the final table was tuned by the same rule.
    _, best_k, predict_buffer, buffer_sweep_df = fit_safety_buffer(train, val, BASE_FEATURES)
    buffer_sweep_df.to_csv(os.path.join(OUTPUT_DIR, "safety_buffer_sweep.csv"), index=False)
    print(f"\n[VAL] Safety-buffer sweep -> chosen k = {best_k:.2f} kg (min total cost)")
    print("[VAL] Safety buffer (LM + k):",
          shortfall_metrics(val['food_waste_kg'].values, predict_buffer(val)))

    _, predict_quantile, best_quantile, quantile_sweep_df = tune_quantile_lgbm(train, val, BASE_FEATURES)
    quantile_sweep_df.to_csv(os.path.join(OUTPUT_DIR, "quantile_lgbm_sweep.csv"), index=False)
    print(f"\n[VAL] Quantile-LGBM sweep -> chosen quantile = {best_quantile:.2f} (min total cost)")
    print("[VAL] Quantile LightGBM:",
          shortfall_metrics(val['food_waste_kg'].values, predict_quantile(val)))

    # ---------- Model selection on VALIDATION only ----------
    # 7a. Penalty sweep, extended out to w=50 to find where the trade-off
    #     plateaus (this replaces the notebook's test-set penalty search).
    #     We also track RMSE at each penalty so the cost of shortfall
    #     reduction, in accuracy terms, is visible -- this is the
    #     cost-benefit curve for the paper.
    penalty_grid = [1, 2, 3, 5, 7, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50]
    penalty_results = []
    for p in penalty_grid:
        _, _, predict_fn = fit_hare(train, BASE_FEATURES, penalty=p)
        pred = predict_fn(val)
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        r = regression_metrics(val['food_waste_kg'].values, pred)
        penalty_results.append({'penalty': p, **m, 'RMSE': r['RMSE'], 'R2': r['R2']})
    penalty_df = pd.DataFrame(penalty_results)

    # Select penalty by TOTAL OPERATIONAL COST, not shortfall mass alone.
    # Shortfall-mass-only selection (the previous version) keeps rewarding
    # bigger penalties forever, because it never "sees" the surplus cost
    # piling up on the other side -- that's why it kept climbing to w=35+
    # while RMSE/R2 kept getting worse for very little extra safety. Cost
    # minimization naturally stops once further shortfall reduction costs
    # more (in wasted surplus) than it saves (in avoided shortfall).
    best_penalty = int(penalty_df.loc[penalty_df['total_cost'].idxmin(), 'penalty'])
    print(f"\n[VAL] Penalty sweep (w = 1..50):")
    print(penalty_df.to_string(index=False))
    print(f"\n[VAL] Chosen penalty (minimizes total operational cost, "
          f"shortfall={COST_PER_KG_SHORTFALL}x, surplus={COST_PER_KG_SURPLUS}x per kg): {best_penalty}")

    # Cost-benefit plot: total operational cost (left axis) vs RMSE (right axis) across w
    fig, ax1 = plt.subplots(figsize=(9, 6))
    ax1.plot(penalty_df['penalty'], penalty_df['total_cost'], 'o-', color='darkred', label='Total operational cost')
    ax1.plot(penalty_df['penalty'], penalty_df['shortfall_mass_kg'], 'x--', color='indianred', alpha=0.6, label='Shortfall mass (kg)')
    ax1.set_xlabel('Under-prediction penalty multiplier (w)')
    ax1.set_ylabel('Cost (red = total cost, faded = shortfall mass kg)', color='darkred')
    ax1.tick_params(axis='y', labelcolor='darkred')
    ax1.axvline(best_penalty, linestyle='--', color='gray', alpha=0.7)
    ax1.text(best_penalty + 0.5, ax1.get_ylim()[1] * 0.9, f'Chosen w={best_penalty} (min cost)', color='gray')
    ax1.legend(loc='upper left', fontsize=8)

    ax2 = ax1.twinx()
    ax2.plot(penalty_df['penalty'], penalty_df['RMSE'], 's--', color='steelblue', label='RMSE')
    ax2.set_ylabel('RMSE (kg) — lower is more accurate', color='steelblue')
    ax2.tick_params(axis='y', labelcolor='steelblue')

    plt.title("Operational Cost vs. Accuracy Trade-off\nPenalty chosen at the point of minimum total cost, not maximum shortfall reduction")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "penalty_cost_benefit_curve.png"), dpi=120)
    plt.close()
    penalty_df.to_csv(os.path.join(OUTPUT_DIR, "penalty_sweep_results.csv"), index=False)

    # 7b. Hyperparameter tuning (train-internal CV, no leakage) at the chosen penalty
    best_params = tune_ghost_hyperparams(train, BASE_FEATURES, best_penalty)
    print(f"\nBest LightGBM ghost params: {best_params}")

    # 7c. Compare feature sets on VALIDATION: base -> extended -> extended+friend's features
    _, _, predict_base_feats = fit_hare(train, BASE_FEATURES, best_penalty, best_params)
    train_fe = engineer_features(train)  # already engineered; kept for clarity
    _, _, predict_ghost_feats = fit_hare(train_fe, GHOST_FEATURES, best_penalty, best_params)
    _, _, predict_ghost_feats_v2 = fit_hare(train_fe, GHOST_FEATURES_V2, best_penalty, best_params)

    val_metrics_basefeat = shortfall_metrics(val['food_waste_kg'].values, predict_base_feats(val))
    val_metrics_ghostfeat = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats(val))
    val_metrics_ghostfeat_v2 = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats_v2(val))
    print("\n[VAL] HARE (base features):", val_metrics_basefeat)
    print("[VAL] HARE (extended ghost features):", val_metrics_ghostfeat)
    print("[VAL] HARE (extended + friend's features):", val_metrics_ghostfeat_v2)

    feature_options = {
        'base': (BASE_FEATURES, val_metrics_basefeat['total_cost']),
        'extended': (GHOST_FEATURES, val_metrics_ghostfeat['total_cost']),
        'extended_v2': (GHOST_FEATURES_V2, val_metrics_ghostfeat_v2['total_cost']),
    }
    best_feature_name = min(feature_options, key=lambda k: feature_options[k][1])
    final_features = feature_options[best_feature_name][0]
    print(f"\n>>> Selected feature set (min total cost on val): {best_feature_name}")

    # 7d. Compare ghost-layer backends (from friend's code: XGBoost, CatBoost)
    # at the chosen penalty + feature set. Still asymmetrically weighted --
    # this tests which tree library learns the weighted residual best, not
    # whether the asymmetric weighting itself helps (that's settled above).
    ghost_options = {}
    for ghost_type in ['lightgbm', 'xgboost', 'catboost']:
        try:
            _, _, predict_gt = fit_hare(train, final_features, best_penalty, ghost_type=ghost_type)
            m = shortfall_metrics(val['food_waste_kg'].values, predict_gt(val))
            ghost_options[ghost_type] = m['total_cost']
            print(f"[VAL] Ghost={ghost_type}: total_cost={m['total_cost']:.2f}  "
                  f"shortfall_mass={m['shortfall_mass_kg']:.2f}  surplus_mass={m['surplus_mass_kg']:.2f}")
        except ImportError:
            print(f"[VAL] Ghost={ghost_type}: skipped (library not installed)")
    best_ghost_type = min(ghost_options, key=ghost_options.get) if ghost_options else 'lightgbm'
    print(f"\n>>> Selected ghost backend (min total cost on val): {best_ghost_type}")

    # Re-tune hyperparameters for the winning backend only if it's LightGBM
    # (tune_ghost_hyperparams is LightGBM-specific; XGBoost/CatBoost use
    # their own defaults here -- extend tune_ghost_hyperparams similarly
    # if one of them wins consistently across datasets).
    if best_ghost_type != 'lightgbm':
        best_params = None

    # ---------- FINAL MODEL: refit on train, touch test exactly once ----------
    final_base, final_ghost, final_predict = fit_hare(
        train, final_features, best_penalty, best_params, ghost_type=best_ghost_type
    )

    test_pred_hare = final_predict(test)
    test_pred_lm = lm.predict(test[BASE_FEATURES])
    test_pred_rf = rf.predict(test[BASE_FEATURES])
    test_pred_buffer = predict_buffer(test)
    test_pred_quantile = predict_quantile(test)
    actual = test['food_waste_kg'].values

    print("\n================ FINAL TEST-SET RESULTS (touched once) ================")
    print("Baseline LM:", regression_metrics(actual, test_pred_lm), shortfall_metrics(actual, test_pred_lm))
    print("Random Forest:", regression_metrics(actual, test_pred_rf), shortfall_metrics(actual, test_pred_rf))
    print(f"Safety Buffer (LM+{best_k:.2f}kg):", regression_metrics(actual, test_pred_buffer), shortfall_metrics(actual, test_pred_buffer))
    print(f"Quantile LGBM (q={best_quantile:.2f}):", regression_metrics(actual, test_pred_quantile), shortfall_metrics(actual, test_pred_quantile))
    print("HARE (final):", regression_metrics(actual, test_pred_hare), shortfall_metrics(actual, test_pred_hare))


    # ---------- Cross-validated evidence for the final model (for the paper) ----------
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmse, cv_shortfall_mass = [], []
    # Rolling-origin CV: each fold trains on an earlier chronological block and
    # evaluates on the block right after it -- matches real deployment, where
    # you only ever have the past to predict the future. Do NOT shuffle here.
    full_train_val = pd.concat([train, val], ignore_index=True).sort_values('date').reset_index(drop=True)
    for tr_idx, te_idx in tscv.split(full_train_val):
        fold_train = full_train_val.iloc[tr_idx]
        fold_eval = full_train_val.iloc[te_idx]
        _, _, fold_predict = fit_hare(fold_train, final_features, best_penalty, best_params, ghost_type=best_ghost_type)
        fold_pred = fold_predict(fold_eval)
        cv_rmse.append(np.sqrt(mean_squared_error(fold_eval['food_waste_kg'], fold_pred)))
        cv_shortfall_mass.append(shortfall_mass_scorer(fold_eval['food_waste_kg'].values, fold_pred))

    print(f"\n5-fold CV (train+val, final model): "
          f"RMSE {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}, "
          f"shortfall mass {np.mean(cv_shortfall_mass):.2f} ± {np.std(cv_shortfall_mass):.2f} kg")

    # ---------- SHAP on final ghost model ----------
    explainer = shap.TreeExplainer(final_ghost)
    shap_sample = test[final_features].iloc[:60]
    shap_values = explainer.shap_values(shap_sample)
    shap_imp = pd.DataFrame({
        'Feature': final_features,
        'Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=shap_imp, x='Importance', y='Feature', hue='Feature',
                palette='crest', legend=False)
    plt.title("Final Model — Ghost Layer Feature Attribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_shap.png"), dpi=120)
    plt.close()

    # ---------- Save final comparison table ----------
    final_table = pd.DataFrame([
        {'Model': 'Baseline LM', **regression_metrics(actual, test_pred_lm), **shortfall_metrics(actual, test_pred_lm)},
        {'Model': 'Random Forest', **regression_metrics(actual, test_pred_rf), **shortfall_metrics(actual, test_pred_rf)},
        {'Model': f'Safety Buffer (LM+{best_k:.2f}kg)', **regression_metrics(actual, test_pred_buffer), **shortfall_metrics(actual, test_pred_buffer)},
        {'Model': f'Quantile LGBM (q={best_quantile:.2f})', **regression_metrics(actual, test_pred_quantile), **shortfall_metrics(actual, test_pred_quantile)},
        {'Model': 'HARE (final)', **regression_metrics(actual, test_pred_hare), **shortfall_metrics(actual, test_pred_hare)},
    ])
    final_table.to_csv(os.path.join(OUTPUT_DIR, "final_test_results.csv"), index=False)
    print(f"\nSaved final results to {OUTPUT_DIR}/final_test_results.csv")
    print(final_table.to_string(index=False))

    # ---------- Cost-ratio sensitivity analysis (operational domain) ----------
    # Includes the two control baselines so the sensitivity plot answers
    # "does HARE still win once compared against the obvious simpler fixes?"
    # not just against LM/RF.
    all_shortfall_masses = {
        'Baseline LM': shortfall_metrics(actual, test_pred_lm)['shortfall_mass_kg'],
        'Random Forest': shortfall_metrics(actual, test_pred_rf)['shortfall_mass_kg'],
        f'Safety Buffer (+{best_k:.1f}kg)': shortfall_metrics(actual, test_pred_buffer)['shortfall_mass_kg'],
        f'Quantile LGBM (q={best_quantile:.2f})': shortfall_metrics(actual, test_pred_quantile)['shortfall_mass_kg'],
        'HARE (final)': shortfall_metrics(actual, test_pred_hare)['shortfall_mass_kg'],
    }
    all_surplus_masses = {
        'Baseline LM': shortfall_metrics(actual, test_pred_lm)['surplus_mass_kg'],
        'Random Forest': shortfall_metrics(actual, test_pred_rf)['surplus_mass_kg'],
        f'Safety Buffer (+{best_k:.1f}kg)': shortfall_metrics(actual, test_pred_buffer)['surplus_mass_kg'],
        f'Quantile LGBM (q={best_quantile:.2f})': shortfall_metrics(actual, test_pred_quantile)['surplus_mass_kg'],
        'HARE (final)': shortfall_metrics(actual, test_pred_hare)['surplus_mass_kg'],
    }
    cost_ratio_sensitivity(
        shortfall_masses=all_shortfall_masses,
        surplus_masses=all_surplus_masses,
        reference_model='HARE (final)',
    )



if __name__ == "__main__":
    main()


Train: 547  Val: 182  Test: 182
After outlier trimming -> Train: 527  Val: 173  Test: 177

[VAL] Baseline LM: {'RMSE': np.float64(5.1293755802527485), 'MAE': 4.24598342992096, 'R2': 0.8052973781304028}
[VAL] Random Forest: {'RMSE': np.float64(5.813091839437628), 'MAE': 4.723485034509184, 'R2': 0.749932548223554}

[VAL] Penalty sweep (w = 1..50):
 penalty  critical_shortfalls  shortfall_mass_kg  max_shortfall_kg   bias_kg     RMSE       R2
       1                   27         307.524202         12.411350 -0.883183 5.436126 0.781314
       2                   24         263.005296         10.795804 -1.461071 5.511387 0.775216
       3                   22         260.737032         10.030699 -1.486449 5.465196 0.778968
       5                   17         227.056505         10.469757 -2.031721 5.702897 0.759323
       7                   14         191.136838          9.382837 -2.373201 5.704174 0.759216
      10                   11         170.119535          9.258656 -2.795925 5.913

In [ ]:
# -*- coding: utf-8 -*-
"""Untitled1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1kDOZ5aDR611NqiXttVHG_u20bMsbmTqy
"""

# -*- coding: utf-8 -*-
"""
food_waste_pipeline.py

Leakage-free rewrite of the HARE (base linear model + asymmetric-weighted
LightGBM "ghost" residual layer) food waste prediction pipeline.

KEY DIFFERENCES FROM THE ORIGINAL NOTEBOOK
-------------------------------------------
1. THREE-WAY SPLIT (train / val / test), not two-way.
   - train: fit models
   - val:   compare model variants, pick hyperparameters, pick the
            asymmetric penalty. This replaces every place the original
            notebook re-checked `test_data` after a tweak.
   - test:  touched exactly ONCE, at the very end, to report final numbers.
2. Outlier trimming is fit on TRAIN ONLY (quartiles computed from train),
   then the same bounds are applied to val/test. The original computed
   quartiles on the full dataset before splitting, which lets the
   test set influence which training rows get kept.
3. Only ONE model is carried into the final test evaluation. Model
   selection (which ghost model / feature set / penalty) happens on
   validation and is decided before test is ever touched.
4. Cross-validated performance is reported for the final model in
   addition to the single test-set snapshot, since one split's numbers
   aren't strong evidence for a paper on their own.

Run this the same way you'd run the notebook: put train.csv next to it.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, make_scorer,
)
from scipy.stats import uniform, randint
import lightgbm as lgb
import shap

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "python_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5      # kg under-prediction considered "critical"
KG_PER_MEAL = 0.4            # used to convert kg -> "meals saved" narrative


# ============================================================
# 1. LOAD + BASIC CLEANING (safe to do before splitting: none of
#    this uses target-derived statistics or cross-row aggregates)
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )

    numeric_cols = ['meals_served', 'kitchen_staff', 'temperature_C',
                     'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in numeric_cols:
        data[col] = clean_numeric(data[col])

    data['staff_experience'] = (
        data['staff_experience'].astype(str).str.lower().str.strip()
        .replace('nan', np.nan).fillna('unknown')
    )
    data['waste_category'] = data['waste_category'].astype(str).str.lower().str.strip()

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month

    data.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
    # NOTE: 'date' is kept here (only 'ID' dropped) so a chronological split is
    # possible. 'date' is dropped later, after splitting, right before modeling.
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)


# ============================================================
# 2. SPLIT FIRST -- everything downstream is fit on train only
#
# This is a forecasting problem (predict tomorrow's waste from today's
# conditions), so a RANDOM split lets future days leak into training folds
# that get evaluated against past days -- performance looks better than it
# would in real deployment, where you only ever have the past. We split
# chronologically instead: train on the earliest days, validate on the
# next chunk, test on the most recent chunk. Random split is kept as an
# option for comparison, but chronological is the default and what you
# should report as your primary result.
# ============================================================
def three_way_split(data, val_size=0.2, test_size=0.2, chronological=True):
    if chronological:
        data = data.sort_values('date').reset_index(drop=True)
        n = len(data)
        n_test = int(n * test_size)
        n_val = int(n * val_size)
        n_train = n - n_val - n_test
        train = data.iloc[:n_train]
        val = data.iloc[n_train:n_train + n_val]
        test = data.iloc[n_train + n_val:]
        return (train.reset_index(drop=True),
                val.reset_index(drop=True),
                test.reset_index(drop=True))

    train_val, test = train_test_split(data, test_size=test_size, random_state=SEED)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, random_state=SEED)
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


# ============================================================
# 3. OUTLIER TRIMMING -- bounds learned from TRAIN, applied to all splits
# ============================================================
def fit_outlier_bounds(train, cols):
    bounds = {}
    for col in cols:
        Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return bounds


def apply_outlier_bounds(df, bounds):
    mask = pd.Series(True, index=df.index)
    for col, (lo, hi) in bounds.items():
        mask &= df[col].between(lo, hi)
    return df[mask].reset_index(drop=True)


# ============================================================
# 4. FEATURE ENGINEERING -- pure row-wise transforms, safe to reuse
#    across splits since nothing is fit here (no leakage risk)
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg",
    "special_event", "temperature_C", "humidity_percent",
    "day_of_week", "staff_efficiency", "demand_pressure",
    "env_score", "event_risk", "waste_ratio",
    "is_weekend", "day", "month",
]

GHOST_EXTRA_FEATURES = [
    'meals_x_humidity', 'past_waste_x_temp', 'kitchen_x_temp',
    'day_x_humidity', 'meals_served_sq', 'past_waste_kg_sq', 'temperature_C_sq',
]
GHOST_FEATURES = BASE_FEATURES + GHOST_EXTRA_FEATURES


def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / df['kitchen_staff']
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['waste_ratio'] = df['past_waste_kg'] / (df['past_waste_kg'] + 1)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['kitchen_x_temp'] = df['kitchen_staff'] * df['temperature_C']
    df['day_x_humidity'] = df['day'] * df['humidity_percent']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['past_waste_kg_sq'] = df['past_waste_kg'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2
    return df


# ============================================================
# 5. MODELS
# ============================================================
def fit_baseline_lm(train):
    lm = LinearRegression().fit(train[BASE_FEATURES], train['food_waste_kg'])
    return lm


def fit_random_forest(train):
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf.fit(train[BASE_FEATURES], train['food_waste_kg'])
    return rf


def shortfall_mass_scorer(y_true, y_pred):
    """Custom scorer: total kg of dangerous under-prediction. Lower is better,
    so RandomizedSearchCV needs the negative of this."""
    resid = y_true - y_pred
    return np.sum(resid[resid > 0])


critical_shortfall_neg_scorer = make_scorer(
    lambda y_true, y_pred: -shortfall_mass_scorer(y_true, y_pred),
    greater_is_better=True,
)


def fit_hare(train, features, penalty, lgbm_params=None):
    """Base linear learner + asymmetric-weighted LightGBM ghost layer
    that learns the base learner's residuals."""
    base = LinearRegression().fit(train[features], train['food_waste_kg'])
    resid = train['food_waste_kg'].values - base.predict(train[features])
    weights = np.where(resid > 0, penalty, 1.0)

    params = dict(n_estimators=300, learning_rate=0.03, random_state=SEED, verbose=-1)
    if lgbm_params:
        params.update(lgbm_params)  # tuned values (incl. n_estimators/learning_rate) now kept

    ghost = lgb.LGBMRegressor(**params)
    ghost.fit(train[features], resid, sample_weight=weights)

    def predict(df):
        return base.predict(df[features]) + ghost.predict(df[features])

    return base, ghost, predict


from sklearn.base import BaseEstimator, RegressorMixin, clone


class HAREEstimator(BaseEstimator, RegressorMixin):
    """Wraps 'fit linear base -> compute residual -> fit weighted ghost on
    residual -> predict base+ghost' as a SINGLE estimator with the real
    target (food_waste_kg), not a pre-computed residual.

    This matters for cross-validated hyperparameter search: if you instead
    precompute residuals once on the full training set and hand them to
    RandomizedSearchCV, the base model has already seen every row -- including
    the rows that later become each fold's held-out validation slice -- before
    tuning even starts. Wrapping it like this forces the base model to be
    refit on ONLY each fold's training rows, so no fold sees its own
    validation data at any stage.
    """

    def __init__(self, penalty=20.0, n_estimators=300, learning_rate=0.03,
                 num_leaves=31, max_depth=-1, min_child_samples=20,
                 subsample=1.0, colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=0.0):
        self.penalty = penalty
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        self.base_ = LinearRegression().fit(X, y)
        resid = y - self.base_.predict(X)
        weights = np.where(resid > 0, self.penalty, 1.0)
        self.ghost_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, max_depth=self.max_depth,
            min_child_samples=self.min_child_samples, subsample=self.subsample,
            colsample_bytree=self.colsample_bytree, reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda, random_state=SEED, verbose=-1,
        )
        self.ghost_.fit(X, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        return self.base_.predict(X) + self.ghost_.predict(X)


def tune_ghost_hyperparams(train, features, penalty, n_iter=60):
    """Hyperparameter search where base+ghost are refit fresh inside every
    CV fold (see HAREEstimator), scored against the real target so the
    asymmetric penalty's effect on shortfalls is part of what's being tuned."""
    param_dist = {
        'n_estimators': randint(100, 1000),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 60),
        'max_depth': randint(5, 15),
        'min_child_samples': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0, 0.5),
    }
    search = RandomizedSearchCV(
        estimator=HAREEstimator(penalty=penalty),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=critical_shortfall_neg_scorer,  # tune toward low shortfall mass, not just RMSE
        cv=5, random_state=SEED, n_jobs=-1,
    )
    search.fit(train[features], train['food_waste_kg'])
    best = {k: v for k, v in search.best_params_.items()}
    return best


# ============================================================
# 6. METRICS
# ============================================================
def regression_metrics(actual, pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
    }


def shortfall_metrics(actual, pred, threshold=SHORTFALL_THRESHOLD):
    resid = actual - pred
    return {
        'critical_shortfalls': int(np.sum(resid > threshold)),
        'shortfall_mass_kg': float(np.sum(resid[resid > 0])),
        'max_shortfall_kg': float(np.max(resid)),
        'bias_kg': float(np.mean(resid)),
    }


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    bounds = fit_outlier_bounds(train, outlier_cols)
    train = apply_outlier_bounds(train, bounds)
    val = apply_outlier_bounds(val, bounds)
    test = apply_outlier_bounds(test, bounds)
    print(f"After outlier trimming -> Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    # ---------- Baselines ----------
    lm = fit_baseline_lm(train)
    rf = fit_random_forest(train)

    val_pred_lm = lm.predict(val[BASE_FEATURES])
    val_pred_rf = rf.predict(val[BASE_FEATURES])
    print("\n[VAL] Baseline LM:", regression_metrics(val['food_waste_kg'], val_pred_lm))
    print("[VAL] Random Forest:", regression_metrics(val['food_waste_kg'], val_pred_rf))

    # ---------- Model selection on VALIDATION only ----------
    # 7a. Penalty sweep, extended out to w=50 to find where the trade-off
    #     plateaus (this replaces the notebook's test-set penalty search).
    #     We also track RMSE at each penalty so the cost of shortfall
    #     reduction, in accuracy terms, is visible -- this is the
    #     cost-benefit curve for the paper.
    penalty_grid = [1, 2, 3, 5, 7, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50]
    penalty_results = []
    for p in penalty_grid:
        _, _, predict_fn = fit_hare(train, BASE_FEATURES, penalty=p)
        pred = predict_fn(val)
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        r = regression_metrics(val['food_waste_kg'].values, pred)
        penalty_results.append({'penalty': p, **m, 'RMSE': r['RMSE'], 'R2': r['R2']})
    penalty_df = pd.DataFrame(penalty_results)

    # Diminishing-returns selection instead of blind argmin: pick the smallest
    # penalty whose shortfall mass is within 5% of the best mass achieved
    # anywhere in the grid. Blind argmin would always land on w=50 even when
    # w=15 gets 95% of the benefit for a fraction of the accuracy cost.
    best_mass = penalty_df['shortfall_mass_kg'].min()
    within_5pct = penalty_df[penalty_df['shortfall_mass_kg'] <= best_mass * 1.05]
    best_penalty = int(within_5pct['penalty'].min())
    print(f"\n[VAL] Penalty sweep (w = 1..50):")
    print(penalty_df.to_string(index=False))
    print(f"\n[VAL] Chosen penalty (smallest w within 5% of best shortfall mass): {best_penalty}")

    # Cost-benefit plot: shortfall mass (left axis) vs RMSE (right axis) across w
    fig, ax1 = plt.subplots(figsize=(9, 6))
    ax1.plot(penalty_df['penalty'], penalty_df['shortfall_mass_kg'], 'o-', color='darkred', label='Shortfall mass (kg)')
    ax1.set_xlabel('Under-prediction penalty multiplier (w)')
    ax1.set_ylabel('Shortfall mass (kg) — lower is safer', color='darkred')
    ax1.tick_params(axis='y', labelcolor='darkred')
    ax1.axvline(best_penalty, linestyle='--', color='gray', alpha=0.7)
    ax1.text(best_penalty + 0.5, ax1.get_ylim()[1] * 0.9, f'Chosen w={best_penalty}', color='gray')

    ax2 = ax1.twinx()
    ax2.plot(penalty_df['penalty'], penalty_df['RMSE'], 's--', color='steelblue', label='RMSE')
    ax2.set_ylabel('RMSE (kg) — lower is more accurate', color='steelblue')
    ax2.tick_params(axis='y', labelcolor='steelblue')

    plt.title("Operational Safety vs. Accuracy Trade-off\nShortfall risk (red) falls as accuracy (blue) worsens with increasing penalty")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "penalty_cost_benefit_curve.png"), dpi=120)
    plt.close()
    penalty_df.to_csv(os.path.join(OUTPUT_DIR, "penalty_sweep_results.csv"), index=False)

    # 7b. Hyperparameter tuning (train-internal CV, no leakage) at the chosen penalty
    best_params = tune_ghost_hyperparams(train, BASE_FEATURES, best_penalty)
    print(f"\nBest LightGBM ghost params: {best_params}")

    # 7c. Compare feature sets on VALIDATION: base features vs. extended ghost features
    _, _, predict_base_feats = fit_hare(train, BASE_FEATURES, best_penalty, best_params)
    train_fe = engineer_features(train)  # already engineered; kept for clarity
    _, _, predict_ghost_feats = fit_hare(train_fe, GHOST_FEATURES, best_penalty, best_params)

    val_metrics_basefeat = shortfall_metrics(val['food_waste_kg'].values, predict_base_feats(val))
    val_metrics_ghostfeat = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats(val))
    print("\n[VAL] HARE (base features):", val_metrics_basefeat)
    print("[VAL] HARE (extended ghost features):", val_metrics_ghostfeat)

    use_extended = val_metrics_ghostfeat['shortfall_mass_kg'] < val_metrics_basefeat['shortfall_mass_kg']
    final_features = GHOST_FEATURES if use_extended else BASE_FEATURES
    print(f"\n>>> Selected feature set: {'extended' if use_extended else 'base'}")

    # ---------- FINAL MODEL: refit on train, touch test exactly once ----------
    final_base, final_ghost, final_predict = fit_hare(train, final_features, best_penalty, best_params)

    test_pred_hare = final_predict(test)
    test_pred_lm = lm.predict(test[BASE_FEATURES])
    test_pred_rf = rf.predict(test[BASE_FEATURES])
    actual = test['food_waste_kg'].values

    print("\n================ FINAL TEST-SET RESULTS (touched once) ================")
    print("Baseline LM:", regression_metrics(actual, test_pred_lm), shortfall_metrics(actual, test_pred_lm))
    print("Random Forest:", regression_metrics(actual, test_pred_rf), shortfall_metrics(actual, test_pred_rf))
    print("HARE (final):", regression_metrics(actual, test_pred_hare), shortfall_metrics(actual, test_pred_hare))

    # ---------- Cross-validated evidence for the final model (for the paper) ----------
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmse, cv_shortfall_mass = [], []
    # Rolling-origin CV: each fold trains on an earlier chronological block and
    # evaluates on the block right after it -- matches real deployment, where
    # you only ever have the past to predict the future. Do NOT shuffle here.
    full_train_val = pd.concat([train, val], ignore_index=True).sort_values('date').reset_index(drop=True)
    for tr_idx, te_idx in tscv.split(full_train_val):
        fold_train = full_train_val.iloc[tr_idx]
        fold_eval = full_train_val.iloc[te_idx]
        _, _, fold_predict = fit_hare(fold_train, final_features, best_penalty, best_params)
        fold_pred = fold_predict(fold_eval)
        cv_rmse.append(np.sqrt(mean_squared_error(fold_eval['food_waste_kg'], fold_pred)))
        cv_shortfall_mass.append(shortfall_mass_scorer(fold_eval['food_waste_kg'].values, fold_pred))

    print(f"\n5-fold CV (train+val, final model): "
          f"RMSE {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}, "
          f"shortfall mass {np.mean(cv_shortfall_mass):.2f} ± {np.std(cv_shortfall_mass):.2f} kg")

    # ---------- SHAP on final ghost model ----------
    explainer = shap.TreeExplainer(final_ghost)
    shap_sample = test[final_features].iloc[:60]
    shap_values = explainer.shap_values(shap_sample)
    shap_imp = pd.DataFrame({
        'Feature': final_features,
        'Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=shap_imp, x='Importance', y='Feature', hue='Feature',
                palette='crest', legend=False)
    plt.title("Final Model — Ghost Layer Feature Attribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_shap.png"), dpi=120)
    plt.close()

    # ---------- Save final comparison table ----------
    final_table = pd.DataFrame([
        {'Model': 'Baseline LM', **regression_metrics(actual, test_pred_lm), **shortfall_metrics(actual, test_pred_lm)},
        {'Model': 'Random Forest', **regression_metrics(actual, test_pred_rf), **shortfall_metrics(actual, test_pred_rf)},
        {'Model': 'HARE (final)', **regression_metrics(actual, test_pred_hare), **shortfall_metrics(actual, test_pred_hare)},
    ])
    final_table.to_csv(os.path.join(OUTPUT_DIR, "final_test_results.csv"), index=False)
    print(f"\nSaved final results to {OUTPUT_DIR}/final_test_results.csv")
    print(final_table.to_string(index=False))


if __name__ == "__main__":
    main()

# -*- coding: utf-8 -*-
"""
food_waste_pipeline.py

Leakage-free rewrite of the HARE (base linear model + asymmetric-weighted
LightGBM "ghost" residual layer) food waste prediction pipeline.

KEY DIFFERENCES FROM THE ORIGINAL NOTEBOOK
-------------------------------------------
1. THREE-WAY SPLIT (train / val / test), not two-way.
   - train: fit models
   - val:   compare model variants, pick hyperparameters, pick the
            asymmetric penalty. This replaces every place the original
            notebook re-checked `test_data` after a tweak.
   - test:  touched exactly ONCE, at the very end, to report final numbers.
2. Outlier trimming is fit on TRAIN ONLY (quartiles computed from train),
   then the same bounds are applied to val/test. The original computed
   quartiles on the full dataset before splitting, which lets the
   test set influence which training rows get kept.
3. Only ONE model is carried into the final test evaluation. Model
   selection (which ghost model / feature set / penalty) happens on
   validation and is decided before test is ever touched.
4. Cross-validated performance is reported for the final model in
   addition to the single test-set snapshot, since one split's numbers
   aren't strong evidence for a paper on their own.

Run this the same way you'd run the notebook: put train.csv next to it.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, make_scorer,
)
from scipy.stats import uniform, randint
import lightgbm as lgb
import shap

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "python_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5      # kg under-prediction considered "critical"
KG_PER_MEAL = 0.4            # used to convert kg -> "meals saved" narrative


# ============================================================
# 1. LOAD + BASIC CLEANING (safe to do before splitting: none of
#    this uses target-derived statistics or cross-row aggregates)
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )

    numeric_cols = ['meals_served', 'kitchen_staff', 'temperature_C',
                     'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in numeric_cols:
        data[col] = clean_numeric(data[col])

    data['staff_experience'] = (
        data['staff_experience'].astype(str).str.lower().str.strip()
        .replace('nan', np.nan).fillna('unknown')
    )
    data['waste_category'] = data['waste_category'].astype(str).str.lower().str.strip()

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month

    data.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
    # NOTE: 'date' is kept here (only 'ID' dropped) so a chronological split is
    # possible. 'date' is dropped later, after splitting, right before modeling.
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)


# ============================================================
# 2. SPLIT FIRST -- everything downstream is fit on train only
#
# This is a forecasting problem (predict tomorrow's waste from today's
# conditions), so a RANDOM split lets future days leak into training folds
# that get evaluated against past days -- performance looks better than it
# would in real deployment, where you only ever have the past. We split
# chronologically instead: train on the earliest days, validate on the
# next chunk, test on the most recent chunk. Random split is kept as an
# option for comparison, but chronological is the default and what you
# should report as your primary result.
# ============================================================
def three_way_split(data, val_size=0.2, test_size=0.2, chronological=True):
    if chronological:
        data = data.sort_values('date').reset_index(drop=True)
        n = len(data)
        n_test = int(n * test_size)
        n_val = int(n * val_size)
        n_train = n - n_val - n_test
        train = data.iloc[:n_train]
        val = data.iloc[n_train:n_train + n_val]
        test = data.iloc[n_train + n_val:]
        return (train.reset_index(drop=True),
                val.reset_index(drop=True),
                test.reset_index(drop=True))

    train_val, test = train_test_split(data, test_size=test_size, random_state=SEED)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, random_state=SEED)
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


# ============================================================
# 3. OUTLIER TRIMMING -- bounds learned from TRAIN, applied to all splits
# ============================================================
def fit_outlier_bounds(train, cols):
    bounds = {}
    for col in cols:
        Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return bounds


def apply_outlier_bounds(df, bounds):
    mask = pd.Series(True, index=df.index)
    for col, (lo, hi) in bounds.items():
        mask &= df[col].between(lo, hi)
    return df[mask].reset_index(drop=True)


# ============================================================
# 4. FEATURE ENGINEERING -- pure row-wise transforms, safe to reuse
#    across splits since nothing is fit here (no leakage risk)
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg",
    "special_event", "temperature_C", "humidity_percent",
    "day_of_week", "staff_efficiency", "demand_pressure",
    "env_score", "event_risk", "waste_ratio",
    "is_weekend", "day", "month",
]

GHOST_EXTRA_FEATURES = [
    'meals_x_humidity', 'past_waste_x_temp', 'kitchen_x_temp',
    'day_x_humidity', 'meals_served_sq', 'past_waste_kg_sq', 'temperature_C_sq',
]
GHOST_FEATURES = BASE_FEATURES + GHOST_EXTRA_FEATURES


def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / df['kitchen_staff']
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['waste_ratio'] = df['past_waste_kg'] / (df['past_waste_kg'] + 1)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['kitchen_x_temp'] = df['kitchen_staff'] * df['temperature_C']
    df['day_x_humidity'] = df['day'] * df['humidity_percent']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['past_waste_kg_sq'] = df['past_waste_kg'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2
    return df


# ============================================================
# 5. MODELS
# ============================================================
def fit_baseline_lm(train):
    lm = LinearRegression().fit(train[BASE_FEATURES], train['food_waste_kg'])
    return lm


def fit_random_forest(train):
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf.fit(train[BASE_FEATURES], train['food_waste_kg'])
    return rf


def shortfall_mass_scorer(y_true, y_pred):
    """Custom scorer: total kg of dangerous under-prediction. Lower is better,
    so RandomizedSearchCV needs the negative of this."""
    resid = y_true - y_pred
    return np.sum(resid[resid > 0])


critical_shortfall_neg_scorer = make_scorer(
    lambda y_true, y_pred: -shortfall_mass_scorer(y_true, y_pred),
    greater_is_better=True,
)


def fit_hare(train, features, penalty, lgbm_params=None):
    """Base linear learner + asymmetric-weighted LightGBM ghost layer
    that learns the base learner's residuals."""
    base = LinearRegression().fit(train[features], train['food_waste_kg'])
    resid = train['food_waste_kg'].values - base.predict(train[features])
    weights = np.where(resid > 0, penalty, 1.0)

    params = dict(n_estimators=300, learning_rate=0.03, random_state=SEED, verbose=-1)
    if lgbm_params:
        params.update(lgbm_params)  # tuned values (incl. n_estimators/learning_rate) now kept

    ghost = lgb.LGBMRegressor(**params)
    ghost.fit(train[features], resid, sample_weight=weights)

    def predict(df):
        return base.predict(df[features]) + ghost.predict(df[features])

    return base, ghost, predict


from sklearn.base import BaseEstimator, RegressorMixin, clone


class HAREEstimator(BaseEstimator, RegressorMixin):
    """Wraps 'fit linear base -> compute residual -> fit weighted ghost on
    residual -> predict base+ghost' as a SINGLE estimator with the real
    target (food_waste_kg), not a pre-computed residual.

    This matters for cross-validated hyperparameter search: if you instead
    precompute residuals once on the full training set and hand them to
    RandomizedSearchCV, the base model has already seen every row -- including
    the rows that later become each fold's held-out validation slice -- before
    tuning even starts. Wrapping it like this forces the base model to be
    refit on ONLY each fold's training rows, so no fold sees its own
    validation data at any stage.
    """

    def __init__(self, penalty=20.0, n_estimators=300, learning_rate=0.03,
                 num_leaves=31, max_depth=-1, min_child_samples=20,
                 subsample=1.0, colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=0.0):
        self.penalty = penalty
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        self.base_ = LinearRegression().fit(X, y)
        resid = y - self.base_.predict(X)
        weights = np.where(resid > 0, self.penalty, 1.0)
        self.ghost_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, max_depth=self.max_depth,
            min_child_samples=self.min_child_samples, subsample=self.subsample,
            colsample_bytree=self.colsample_bytree, reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda, random_state=SEED, verbose=-1,
        )
        self.ghost_.fit(X, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        return self.base_.predict(X) + self.ghost_.predict(X)


def tune_ghost_hyperparams(train, features, penalty, n_iter=60):
    """Hyperparameter search where base+ghost are refit fresh inside every
    CV fold (see HAREEstimator), scored against the real target so the
    asymmetric penalty's effect on shortfalls is part of what's being tuned."""
    param_dist = {
        'n_estimators': randint(100, 1000),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 60),
        'max_depth': randint(5, 15),
        'min_child_samples': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0, 0.5),
    }
    search = RandomizedSearchCV(
        estimator=HAREEstimator(penalty=penalty),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=critical_shortfall_neg_scorer,  # tune toward low shortfall mass, not just RMSE
        cv=5, random_state=SEED, n_jobs=-1,
    )
    search.fit(train[features], train['food_waste_kg'])
    best = {k: v for k, v in search.best_params_.items()}
    return best


# ============================================================
# 6. METRICS
# ============================================================
def regression_metrics(actual, pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
    }


def shortfall_metrics(actual, pred, threshold=SHORTFALL_THRESHOLD):
    resid = actual - pred
    return {
        'critical_shortfalls': int(np.sum(resid > threshold)),
        'shortfall_mass_kg': float(np.sum(resid[resid > 0])),
        'max_shortfall_kg': float(np.max(resid)),
        'bias_kg': float(np.mean(resid)),
    }


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    bounds = fit_outlier_bounds(train, outlier_cols)
    train = apply_outlier_bounds(train, bounds)
    val = apply_outlier_bounds(val, bounds)
    test = apply_outlier_bounds(test, bounds)
    print(f"After outlier trimming -> Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    # ---------- Baselines ----------
    lm = fit_baseline_lm(train)
    rf = fit_random_forest(train)

    val_pred_lm = lm.predict(val[BASE_FEATURES])
    val_pred_rf = rf.predict(val[BASE_FEATURES])
    print("\n[VAL] Baseline LM:", regression_metrics(val['food_waste_kg'], val_pred_lm))
    print("[VAL] Random Forest:", regression_metrics(val['food_waste_kg'], val_pred_rf))

    # ---------- Model selection on VALIDATION only ----------
    # 7a. Penalty sweep (this replaces the notebook's test-set penalty search)
    penalty_grid = [1, 2, 3, 5, 7, 10, 12, 15, 20]
    penalty_results = []
    for p in penalty_grid:
        _, _, predict_fn = fit_hare(train, BASE_FEATURES, penalty=p)
        m = shortfall_metrics(val['food_waste_kg'].values, predict_fn(val))
        penalty_results.append({'penalty': p, **m})
    penalty_df = pd.DataFrame(penalty_results)
    best_penalty = int(penalty_df.loc[penalty_df['shortfall_mass_kg'].idxmin(), 'penalty'])
    print(f"\n[VAL] Chosen penalty (min shortfall mass on val): {best_penalty}")
    print(penalty_df.to_string(index=False))

    # 7b. Hyperparameter tuning (train-internal CV, no leakage) at the chosen penalty
    best_params = tune_ghost_hyperparams(train, BASE_FEATURES, best_penalty)
    print(f"\nBest LightGBM ghost params: {best_params}")

    # 7c. Compare feature sets on VALIDATION: base features vs. extended ghost features
    _, _, predict_base_feats = fit_hare(train, BASE_FEATURES, best_penalty, best_params)
    train_fe = engineer_features(train)  # already engineered; kept for clarity
    _, _, predict_ghost_feats = fit_hare(train_fe, GHOST_FEATURES, best_penalty, best_params)

    val_metrics_basefeat = shortfall_metrics(val['food_waste_kg'].values, predict_base_feats(val))
    val_metrics_ghostfeat = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats(val))
    print("\n[VAL] HARE (base features):", val_metrics_basefeat)
    print("[VAL] HARE (extended ghost features):", val_metrics_ghostfeat)

    use_extended = val_metrics_ghostfeat['shortfall_mass_kg'] < val_metrics_basefeat['shortfall_mass_kg']
    final_features = GHOST_FEATURES if use_extended else BASE_FEATURES
    print(f"\n>>> Selected feature set: {'extended' if use_extended else 'base'}")

    # ---------- FINAL MODEL: refit on train, touch test exactly once ----------
    final_base, final_ghost, final_predict = fit_hare(train, final_features, best_penalty, best_params)

    test_pred_hare = final_predict(test)
    test_pred_lm = lm.predict(test[BASE_FEATURES])
    test_pred_rf = rf.predict(test[BASE_FEATURES])
    actual = test['food_waste_kg'].values

    print("\n================ FINAL TEST-SET RESULTS (touched once) ================")
    print("Baseline LM:", regression_metrics(actual, test_pred_lm), shortfall_metrics(actual, test_pred_lm))
    print("Random Forest:", regression_metrics(actual, test_pred_rf), shortfall_metrics(actual, test_pred_rf))
    print("HARE (final):", regression_metrics(actual, test_pred_hare), shortfall_metrics(actual, test_pred_hare))

    # ---------- Cross-validated evidence for the final model (for the paper) ----------
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmse, cv_shortfall_mass = [], []
    # Rolling-origin CV: each fold trains on an earlier chronological block and
    # evaluates on the block right after it -- matches real deployment, where
    # you only ever have the past to predict the future. Do NOT shuffle here.
    full_train_val = pd.concat([train, val], ignore_index=True).sort_values('date').reset_index(drop=True)
    for tr_idx, te_idx in tscv.split(full_train_val):
        fold_train = full_train_val.iloc[tr_idx]
        fold_eval = full_train_val.iloc[te_idx]
        _, _, fold_predict = fit_hare(fold_train, final_features, best_penalty, best_params)
        fold_pred = fold_predict(fold_eval)
        cv_rmse.append(np.sqrt(mean_squared_error(fold_eval['food_waste_kg'], fold_pred)))
        cv_shortfall_mass.append(shortfall_mass_scorer(fold_eval['food_waste_kg'].values, fold_pred))

    print(f"\n5-fold CV (train+val, final model): "
          f"RMSE {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}, "
          f"shortfall mass {np.mean(cv_shortfall_mass):.2f} ± {np.std(cv_shortfall_mass):.2f} kg")

    # ---------- SHAP on final ghost model ----------
    explainer = shap.TreeExplainer(final_ghost)
    shap_sample = test[final_features].iloc[:60]
    shap_values = explainer.shap_values(shap_sample)
    shap_imp = pd.DataFrame({
        'Feature': final_features,
        'Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=shap_imp, x='Importance', y='Feature', hue='Feature',
                palette='crest', legend=False)
    plt.title("Final Model — Ghost Layer Feature Attribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_shap.png"), dpi=120)
    plt.close()

    # ---------- Save final comparison table ----------
    final_table = pd.DataFrame([
        {'Model': 'Baseline LM', **regression_metrics(actual, test_pred_lm), **shortfall_metrics(actual, test_pred_lm)},
        {'Model': 'Random Forest', **regression_metrics(actual, test_pred_rf), **shortfall_metrics(actual, test_pred_rf)},
        {'Model': 'HARE (final)', **regression_metrics(actual, test_pred_hare), **shortfall_metrics(actual, test_pred_hare)},
    ])
    final_table.to_csv(os.path.join(OUTPUT_DIR, "final_test_results.csv"), index=False)
    print(f"\nSaved final results to {OUTPUT_DIR}/final_test_results.csv")
    print(final_table.to_string(index=False))


if __name__ == "__main__":
    main()

# -*- coding: utf-8 -*-
"""
food_waste_pipeline.py

Leakage-free rewrite of the HARE (base linear model + asymmetric-weighted
LightGBM "ghost" residual layer) food waste prediction pipeline.

KEY DIFFERENCES FROM THE ORIGINAL NOTEBOOK
-------------------------------------------
1. THREE-WAY SPLIT (train / val / test), not two-way.
   - train: fit models
   - val:   compare model variants, pick hyperparameters, pick the
            asymmetric penalty. This replaces every place the original
            notebook re-checked `test_data` after a tweak.
   - test:  touched exactly ONCE, at the very end, to report final numbers.
2. Outlier trimming is fit on TRAIN ONLY (quartiles computed from train),
   then the same bounds are applied to val/test. The original computed
   quartiles on the full dataset before splitting, which lets the
   test set influence which training rows get kept.
3. Only ONE model is carried into the final test evaluation. Model
   selection (which ghost model / feature set / penalty) happens on
   validation and is decided before test is ever touched.
4. Cross-validated performance is reported for the final model in
   addition to the single test-set snapshot, since one split's numbers
   aren't strong evidence for a paper on their own.

Run this the same way you'd run the notebook: put train.csv next to it.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, make_scorer,
)
from scipy.stats import uniform, randint
import lightgbm as lgb
import shap

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "python_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5      # kg under-prediction considered "critical"
KG_PER_MEAL = 0.4            # used to convert kg -> "meals saved" narrative

# ---- OPERATIONAL COST ASSUMPTIONS ------------------------------------
# These turn "kg of shortfall" and "kg of surplus" into a single comparable
# number so we can pick a penalty (and compare models) on cost, not on
# shortfall mass alone or on R2/RMSE alone. The two are NOT the same kind
# of cost: running short means lost sales, refunds, and reputational
# damage; over-preparing just means wasted ingredients. That's WHY the
# asymmetric penalty exists in the first place, but until now nothing
# converted that intuition into one number you could optimize against.
#
# The values below are placeholders (shortfall costs 3x more per kg than
# surplus, a common assumption in newsvendor/inventory literature) --
# replace them with your kitchen's real figures (ingredient cost per kg
# for surplus, and a defensible estimate of lost-sale + reputational cost
# per kg for shortfall) before this goes in the paper. State the
# assumption explicitly in your methods section either way.
COST_PER_KG_SHORTFALL = 3.0
COST_PER_KG_SURPLUS = 1.0


# ============================================================
# 1. LOAD + BASIC CLEANING (safe to do before splitting: none of
#    this uses target-derived statistics or cross-row aggregates)
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )

    numeric_cols = ['meals_served', 'kitchen_staff', 'temperature_C',
                     'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in numeric_cols:
        data[col] = clean_numeric(data[col])

    data['staff_experience'] = (
        data['staff_experience'].astype(str).str.lower().str.strip()
        .replace('nan', np.nan).fillna('unknown')
    )
    data['waste_category'] = data['waste_category'].astype(str).str.lower().str.strip()

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month

    data.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
    # NOTE: 'date' is kept here (only 'ID' dropped) so a chronological split is
    # possible. 'date' is dropped later, after splitting, right before modeling.
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)


# ============================================================
# 2. SPLIT FIRST -- everything downstream is fit on train only
#
# This is a forecasting problem (predict tomorrow's waste from today's
# conditions), so a RANDOM split lets future days leak into training folds
# that get evaluated against past days -- performance looks better than it
# would in real deployment, where you only ever have the past. We split
# chronologically instead: train on the earliest days, validate on the
# next chunk, test on the most recent chunk. Random split is kept as an
# option for comparison, but chronological is the default and what you
# should report as your primary result.
# ============================================================
def three_way_split(data, val_size=0.2, test_size=0.2, chronological=True):
    if chronological:
        data = data.sort_values('date').reset_index(drop=True)
        n = len(data)
        n_test = int(n * test_size)
        n_val = int(n * val_size)
        n_train = n - n_val - n_test
        train = data.iloc[:n_train]
        val = data.iloc[n_train:n_train + n_val]
        test = data.iloc[n_train + n_val:]
        return (train.reset_index(drop=True),
                val.reset_index(drop=True),
                test.reset_index(drop=True))

    train_val, test = train_test_split(data, test_size=test_size, random_state=SEED)
    relative_val = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=relative_val, random_state=SEED)
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


# ============================================================
# 3. OUTLIER TRIMMING -- bounds learned from TRAIN, applied to all splits
# ============================================================
def fit_outlier_bounds(train, cols):
    bounds = {}
    for col in cols:
        Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return bounds


def apply_outlier_bounds(df, bounds):
    mask = pd.Series(True, index=df.index)
    for col, (lo, hi) in bounds.items():
        mask &= df[col].between(lo, hi)
    return df[mask].reset_index(drop=True)


# ============================================================
# 4. FEATURE ENGINEERING -- pure row-wise transforms, safe to reuse
#    across splits since nothing is fit here (no leakage risk)
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg",
    "special_event", "temperature_C", "humidity_percent",
    "day_of_week", "staff_efficiency", "demand_pressure",
    "env_score", "event_risk", "waste_ratio",
    "is_weekend", "day", "month",
]

GHOST_EXTRA_FEATURES = [
    'meals_x_humidity', 'past_waste_x_temp', 'kitchen_x_temp',
    'day_x_humidity', 'meals_served_sq', 'past_waste_kg_sq', 'temperature_C_sq',
]
GHOST_FEATURES = BASE_FEATURES + GHOST_EXTRA_FEATURES

# From the friend's code review: these two are genuinely new (not
# redundant with what's above) and worth testing on validation. Kept as a
# SEPARATE candidate set rather than merged in blind, so the pipeline can
# show on val whether they actually help before they're trusted.
FRIEND_EXTRA_FEATURES = ['temp_humidity_index', 'meal_humidity_ratio']
GHOST_FEATURES_V2 = GHOST_FEATURES + FRIEND_EXTRA_FEATURES


def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / df['kitchen_staff']
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['waste_ratio'] = df['past_waste_kg'] / (df['past_waste_kg'] + 1)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['kitchen_x_temp'] = df['kitchen_staff'] * df['temperature_C']
    df['day_x_humidity'] = df['day'] * df['humidity_percent']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['past_waste_kg_sq'] = df['past_waste_kg'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2

    # Friend's additions (kept separate from the set above; tested, not assumed)
    df['temp_humidity_index'] = df['temperature_C'] + 0.1 * df['humidity_percent']
    df['meal_humidity_ratio'] = df['meals_served'] / (df['humidity_percent'] + 1)
    return df


# ============================================================
# 5. MODELS
# ============================================================
def fit_baseline_lm(train):
    lm = LinearRegression().fit(train[BASE_FEATURES], train['food_waste_kg'])
    return lm


def fit_random_forest(train):
    rf = RandomForestRegressor(n_estimators=500, random_state=SEED)
    rf.fit(train[BASE_FEATURES], train['food_waste_kg'])
    return rf


def shortfall_mass_scorer(y_true, y_pred):
    """Custom scorer: total kg of dangerous under-prediction. Lower is better,
    so RandomizedSearchCV needs the negative of this."""
    resid = y_true - y_pred
    return np.sum(resid[resid > 0])


critical_shortfall_neg_scorer = make_scorer(
    lambda y_true, y_pred: -shortfall_mass_scorer(y_true, y_pred),
    greater_is_better=True,
)


def _make_ghost_model(ghost_type, params, seed=SEED):
    """Build a ghost-layer regressor. All three backends are trained with
    the SAME asymmetric sample_weight -- the point is to compare which
    tree library learns the weighted residual best, not to drop the
    asymmetric weighting the way the unweighted XGBoost version did.

    DETERMINISM: each backend is pinned to single-threaded, bit-reproducible
    training. Fixing random_state alone is NOT enough for gradient-boosted
    trees -- multi-threaded histogram building sums floating-point values in
    a thread-race-dependent order, which changes split points slightly from
    run to run even with an identical seed and identical data. That's why
    the same script produced RMSE 6.700 / 6.573 / 6.676 on three "identical"
    runs. n_jobs=1 (LightGBM/XGBoost) and thread_count=1 (CatBoost) remove
    that source of variance; LightGBM additionally needs deterministic=True
    and force_row_wise=True, since its default multi-threaded histogram
    path can be nondeterministic even at n_jobs=1 without these flags.
    """
    if ghost_type == 'lightgbm':
        base_params = dict(n_estimators=300, learning_rate=0.03, random_state=seed, verbose=-1,
                            n_jobs=1, deterministic=True, force_row_wise=True)
        base_params.update(params or {})
        return lgb.LGBMRegressor(**base_params)
    elif ghost_type == 'xgboost':
        import xgboost as xgb
        base_params = dict(n_estimators=300, learning_rate=0.03, random_state=seed,
                            objective='reg:squarederror', n_jobs=1)
        base_params.update(params or {})
        return xgb.XGBRegressor(**base_params)
    elif ghost_type == 'catboost':
        import catboost as cb
        base_params = dict(iterations=300, learning_rate=0.03, random_seed=seed, verbose=False,
                            thread_count=1)
        base_params.update(params or {})
        return cb.CatBoostRegressor(**base_params)
    else:
        raise ValueError(f"Unknown ghost_type: {ghost_type}")


def fit_hare(train, features, penalty, lgbm_params=None, ghost_type='lightgbm'):
    """Base linear learner + asymmetric-weighted ghost layer (LightGBM by
    default; pass ghost_type='xgboost' or 'catboost' to swap backends)
    that learns the base learner's residuals."""
    base = LinearRegression().fit(train[features], train['food_waste_kg'])
    resid = train['food_waste_kg'].values - base.predict(train[features])
    weights = np.where(resid > 0, penalty, 1.0)

    ghost = _make_ghost_model(ghost_type, lgbm_params)
    ghost.fit(train[features], resid, sample_weight=weights)

    def predict(df):
        return base.predict(df[features]) + ghost.predict(df[features])

    return base, ghost, predict


from sklearn.base import BaseEstimator, RegressorMixin, clone


class HAREEstimator(BaseEstimator, RegressorMixin):
    """Wraps 'fit linear base -> compute residual -> fit weighted ghost on
    residual -> predict base+ghost' as a SINGLE estimator with the real
    target (food_waste_kg), not a pre-computed residual.

    This matters for cross-validated hyperparameter search: if you instead
    precompute residuals once on the full training set and hand them to
    RandomizedSearchCV, the base model has already seen every row -- including
    the rows that later become each fold's held-out validation slice -- before
    tuning even starts. Wrapping it like this forces the base model to be
    refit on ONLY each fold's training rows, so no fold sees its own
    validation data at any stage.
    """

    def __init__(self, penalty=20.0, n_estimators=300, learning_rate=0.03,
                 num_leaves=31, max_depth=-1, min_child_samples=20,
                 subsample=1.0, colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=0.0):
        self.penalty = penalty
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        self.base_ = LinearRegression().fit(X, y)
        resid = y - self.base_.predict(X)
        weights = np.where(resid > 0, self.penalty, 1.0)
        self.ghost_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, max_depth=self.max_depth,
            min_child_samples=self.min_child_samples, subsample=self.subsample,
            colsample_bytree=self.colsample_bytree, reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda, random_state=SEED, verbose=-1,
            # Pinned for determinism -- see _make_ghost_model docstring for why
            # random_state alone doesn't guarantee reproducible LightGBM fits.
            n_jobs=1, deterministic=True, force_row_wise=True,
        )

        self.ghost_.fit(X, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        return self.base_.predict(X) + self.ghost_.predict(X)


def tune_ghost_hyperparams(train, features, penalty, n_iter=60):
    """Hyperparameter search where base+ghost are refit fresh inside every
    CV fold (see HAREEstimator), scored against the real target so the
    asymmetric penalty's effect on shortfalls is part of what's being tuned.

    n_jobs on the search itself is left at -1 (parallelizes across CV folds/
    candidates via joblib, which is safe and deterministic given a fixed
    random_state) -- each individual LGBMRegressor inside HAREEstimator is
    pinned to n_jobs=1 so folds running concurrently don't oversubscribe
    threads or hit LightGBM's nondeterministic multi-threaded path.
    """
    param_dist = {
        'n_estimators': randint(100, 1000),
        'learning_rate': uniform(0.01, 0.1),
        'num_leaves': randint(20, 60),
        'max_depth': randint(5, 15),
        'min_child_samples': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0, 0.5),
    }
    search = RandomizedSearchCV(
        estimator=HAREEstimator(penalty=penalty),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=critical_shortfall_neg_scorer,  # tune toward low shortfall mass, not just RMSE
        cv=5, random_state=SEED, n_jobs=-1,
    )
    search.fit(train[features], train['food_waste_kg'])
    best = {k: v for k, v in search.best_params_.items()}
    return best


# ============================================================
# 6. METRICS
# ============================================================
def regression_metrics(actual, pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
    }


def shortfall_metrics(actual, pred, threshold=SHORTFALL_THRESHOLD):
    resid = actual - pred  # positive = under-prediction (shortfall), negative = over-prediction (surplus)
    shortfall_mass = float(np.sum(resid[resid > 0]))
    surplus_mass = float(np.sum(-resid[resid < 0]))
    total_cost = shortfall_mass * COST_PER_KG_SHORTFALL + surplus_mass * COST_PER_KG_SURPLUS
    return {
        'critical_shortfalls': int(np.sum(resid > threshold)),
        'shortfall_mass_kg': shortfall_mass,
        'surplus_mass_kg': surplus_mass,
        'total_cost': total_cost,
        'max_shortfall_kg': float(np.max(resid)),
        'bias_kg': float(np.mean(resid)),
    }


# ============================================================
# 6b. COST-RATIO SENSITIVITY ANALYSIS (operational domain)
# ============================================================
# Total operational cost is linear in the shortfall:surplus cost ratio r:
#   cost(r) = shortfall_mass_kg * r + surplus_mass_kg          (surplus cost fixed at 1)
# Two models' cost lines cross at exactly one r (the "breakeven ratio"):
#   r* = (surplus_B - surplus_A) / (shortfall_A - shortfall_B)
# This is exact arithmetic on the test-set masses already computed --
# no retraining, no extra assumptions beyond the linear cost model already
# used everywhere else in this pipeline. It answers "how sensitive is the
# 'HARE is best' claim to the COST_PER_KG_SHORTFALL/SURPLUS assumption?"
# explicitly, instead of reporting a single number at one assumed ratio.
def compute_breakeven_ratio(short_a, surplus_a, short_b, surplus_b):
    """r at which cost_a(r) == cost_b(r). Returns None if the lines are
    parallel (equal shortfall mass -> no finite crossing)."""
    denom = short_a - short_b
    if denom == 0:
        return None
    return (surplus_b - surplus_a) / denom


def cost_ratio_sensitivity(shortfall_masses, surplus_masses, reference_model,
                            r_range=(1.0, 5.0), r_step=0.05):
    """
    Sensitivity analysis of total operational cost to the shortfall:surplus
    cost ratio r, for every model in shortfall_masses/surplus_masses.

    Produces:
      - {OUTPUT_DIR}/cost_ratio_sensitivity.csv   (cost curves, all models, all r)
      - {OUTPUT_DIR}/cost_ratio_sensitivity.png   (the sensitivity plot)
      - printed breakeven ratios: reference_model vs every other model

    This does NOT retrain anything -- it is pure post-hoc arithmetic on
    shortfall_mass_kg / surplus_mass_kg already produced by shortfall_metrics().
    Reports the exact breakeven point rather than a single cost number at
    one assumed ratio, so the "best model" claim is shown with its domain
    of validity, not asserted unconditionally.
    """
    models = list(shortfall_masses.keys())
    r_vals = np.arange(r_range[0], r_range[1] + 1e-9, r_step)

    curve_rows = []
    for r in r_vals:
        row = {'r': round(float(r), 4)}
        for m in models:
            row[m] = shortfall_masses[m] * r + surplus_masses[m]
        curve_rows.append(row)
    curve_df = pd.DataFrame(curve_rows)
    curve_df.to_csv(os.path.join(OUTPUT_DIR, "cost_ratio_sensitivity.csv"), index=False)

    breakevens = {}
    print(f"\n================ COST-RATIO SENSITIVITY (reference: {reference_model}) ================")
    for m in models:
        if m == reference_model:
            continue
        r_star = compute_breakeven_ratio(
            shortfall_masses[reference_model], surplus_masses[reference_model],
            shortfall_masses[m], surplus_masses[m],
        )
        breakevens[m] = r_star
        if r_star is None:
            print(f"{reference_model} vs {m}: shortfall masses equal -> no finite breakeven "
                  f"(one model dominates at every r)")
        else:
            direction = ("wins for r >" if shortfall_masses[reference_model] < shortfall_masses[m]
                         else "wins for r <")
            print(f"{reference_model} vs {m}: breakeven r* = {r_star:.3f}  "
                  f"({reference_model} {direction} {r_star:.3f})")

    plt.figure(figsize=(9, 6))
    for m in models:
        style = '-' if m == reference_model else '--'
        lw = 2.5 if m == reference_model else 1.5
        plt.plot(curve_df['r'], curve_df[m], style, linewidth=lw, label=m)
    ymax = curve_df[models].values.max()
    for m, r_star in breakevens.items():
        if r_star is not None and r_range[0] <= r_star <= r_range[1]:
            plt.axvline(r_star, linestyle=':', color='gray', alpha=0.7)
            plt.text(r_star + 0.03, ymax * 0.95, f'r*={r_star:.2f}\n({m})',
                      fontsize=8, color='gray', va='top')
    plt.xlabel('Shortfall-to-surplus cost ratio (r)')
    plt.ylabel('Total operational cost (test set)')
    plt.title(f'Cost-ratio sensitivity: total operational cost vs r\n'
              f'({reference_model} is the reference / proposed method)')
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "cost_ratio_sensitivity.png"), dpi=120)
    plt.close()

    print(f"\nSaved sensitivity curve data to {OUTPUT_DIR}/cost_ratio_sensitivity.csv")
    print(f"Saved sensitivity plot to {OUTPUT_DIR}/cost_ratio_sensitivity.png")

    return curve_df, breakevens


# ============================================================
# 6c. CONTROL BASELINES: constant safety buffer, quantile regression
# ============================================================
# Both answer the same reviewer question from two different angles:
# "Why does HARE need a two-stage residual model at all?"
#
#   - Safety buffer: the cheapest possible fix -- take the linear model and
#     add a constant kg to every prediction. If HARE just reduces to "the
#     linear model plus a blanket offset", a constant shift should match it.
#     Tuned on val by the SAME total-cost criterion as HARE's penalty w,
#     so the comparison is apples-to-apples.
#   - Quantile regression: the standard tool for asymmetric-loss demand
#     prediction in the OR/inventory literature (newsvendor-style). Trained
#     directly on the pinball loss at a chosen quantile, with NO base model
#     or residual step. If quantile LightGBM matches HARE's cost, the
#     two-stage base+ghost architecture isn't earning its complexity.
def fit_safety_buffer(train, val, features, k_grid=None):
    """Baseline LM + constant offset k, tuned on val to minimize total cost
    (same objective HARE's penalty sweep uses). No per-row adaptivity --
    every prediction gets the same additive buffer regardless of that row's
    predicted risk."""
    if k_grid is None:
        k_grid = np.arange(0, 15.25, 0.25)

    lm = LinearRegression().fit(train[features], train['food_waste_kg'])
    val_base_pred = lm.predict(val[features])

    results = []
    for k in k_grid:
        pred = val_base_pred + k
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        results.append({'k': float(k), **m})
    k_df = pd.DataFrame(results)
    best_k = float(k_df.loc[k_df['total_cost'].idxmin(), 'k'])

    def predict(df):
        return lm.predict(df[features]) + best_k

    return lm, best_k, predict, k_df


def fit_quantile_lgbm(train, features, quantile, lgbm_params=None):
    """Single-stage LightGBM trained directly on pinball loss at the given
    quantile -- no base linear model, no residual stage. The standard
    alternative to HARE's architecture for asymmetric-loss prediction."""
    base_params = dict(
        objective='quantile', alpha=quantile,
        n_estimators=300, learning_rate=0.03, random_state=SEED, verbose=-1,
        n_jobs=1, deterministic=True, force_row_wise=True,
    )
    base_params.update(lgbm_params or {})
    model = lgb.LGBMRegressor(**base_params)
    model.fit(train[features], train['food_waste_kg'])

    def predict(df):
        return model.predict(df[features])

    return model, predict


def tune_quantile_lgbm(train, val, features, quantile_grid=None):
    """Sweep the target quantile on val (analogous to HARE's penalty sweep),
    select by total operational cost, using default LightGBM hyperparams
    for a fair "out of the box" comparison against HARE's tuned ghost."""
    if quantile_grid is None:
        quantile_grid = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

    results = []
    models = {}
    for q in quantile_grid:
        model, predict_fn = fit_quantile_lgbm(train, features, q)
        pred = predict_fn(val)
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        results.append({'quantile': q, **m})
        models[q] = (model, predict_fn)
    q_df = pd.DataFrame(results)
    best_q = float(q_df.loc[q_df['total_cost'].idxmin(), 'quantile'])
    return models[best_q][0], models[best_q][1], best_q, q_df


# ============================================================
# 7. MAIN PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    bounds = fit_outlier_bounds(train, outlier_cols)
    train = apply_outlier_bounds(train, bounds)
    val = apply_outlier_bounds(val, bounds)
    test = apply_outlier_bounds(test, bounds)
    print(f"After outlier trimming -> Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    # ---------- Baselines ----------
    lm = fit_baseline_lm(train)
    rf = fit_random_forest(train)

    val_pred_lm = lm.predict(val[BASE_FEATURES])
    val_pred_rf = rf.predict(val[BASE_FEATURES])
    print("\n[VAL] Baseline LM:", regression_metrics(val['food_waste_kg'], val_pred_lm))
    print("[VAL] Random Forest:", regression_metrics(val['food_waste_kg'], val_pred_rf))

    # ---------- Control baselines (chosen on VAL, same as HARE's penalty) ----------
    # These directly answer "why not just do something simpler?" -- selected
    # by the identical total-cost criterion as HARE's penalty w, so every
    # model in the final table was tuned by the same rule.
    _, best_k, predict_buffer, buffer_sweep_df = fit_safety_buffer(train, val, BASE_FEATURES)
    buffer_sweep_df.to_csv(os.path.join(OUTPUT_DIR, "safety_buffer_sweep.csv"), index=False)
    print(f"\n[VAL] Safety-buffer sweep -> chosen k = {best_k:.2f} kg (min total cost)")
    print("[VAL] Safety buffer (LM + k):",
          shortfall_metrics(val['food_waste_kg'].values, predict_buffer(val)))

    _, predict_quantile, best_quantile, quantile_sweep_df = tune_quantile_lgbm(train, val, BASE_FEATURES)
    quantile_sweep_df.to_csv(os.path.join(OUTPUT_DIR, "quantile_lgbm_sweep.csv"), index=False)
    print(f"\n[VAL] Quantile-LGBM sweep -> chosen quantile = {best_quantile:.2f} (min total cost)")
    print("[VAL] Quantile LightGBM:",
          shortfall_metrics(val['food_waste_kg'].values, predict_quantile(val)))

    # ---------- Model selection on VALIDATION only ----------
    # 7a. Penalty sweep, extended out to w=50 to find where the trade-off
    #     plateaus (this replaces the notebook's test-set penalty search).
    #     We also track RMSE at each penalty so the cost of shortfall
    #     reduction, in accuracy terms, is visible -- this is the
    #     cost-benefit curve for the paper.
    penalty_grid = [1, 2, 3, 5, 7, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50]
    penalty_results = []
    for p in penalty_grid:
        _, _, predict_fn = fit_hare(train, BASE_FEATURES, penalty=p)
        pred = predict_fn(val)
        m = shortfall_metrics(val['food_waste_kg'].values, pred)
        r = regression_metrics(val['food_waste_kg'].values, pred)
        penalty_results.append({'penalty': p, **m, 'RMSE': r['RMSE'], 'R2': r['R2']})
    penalty_df = pd.DataFrame(penalty_results)

    # Select penalty by TOTAL OPERATIONAL COST, not shortfall mass alone.
    # Shortfall-mass-only selection (the previous version) keeps rewarding
    # bigger penalties forever, because it never "sees" the surplus cost
    # piling up on the other side -- that's why it kept climbing to w=35+
    # while RMSE/R2 kept getting worse for very little extra safety. Cost
    # minimization naturally stops once further shortfall reduction costs
    # more (in wasted surplus) than it saves (in avoided shortfall).
    best_penalty = int(penalty_df.loc[penalty_df['total_cost'].idxmin(), 'penalty'])
    print(f"\n[VAL] Penalty sweep (w = 1..50):")
    print(penalty_df.to_string(index=False))
    print(f"\n[VAL] Chosen penalty (minimizes total operational cost, "
          f"shortfall={COST_PER_KG_SHORTFALL}x, surplus={COST_PER_KG_SURPLUS}x per kg): {best_penalty}")

    # Cost-benefit plot: total operational cost (left axis) vs RMSE (right axis) across w
    fig, ax1 = plt.subplots(figsize=(9, 6))
    ax1.plot(penalty_df['penalty'], penalty_df['total_cost'], 'o-', color='darkred', label='Total operational cost')
    ax1.plot(penalty_df['penalty'], penalty_df['shortfall_mass_kg'], 'x--', color='indianred', alpha=0.6, label='Shortfall mass (kg)')
    ax1.set_xlabel('Under-prediction penalty multiplier (w)')
    ax1.set_ylabel('Cost (red = total cost, faded = shortfall mass kg)', color='darkred')
    ax1.tick_params(axis='y', labelcolor='darkred')
    ax1.axvline(best_penalty, linestyle='--', color='gray', alpha=0.7)
    ax1.text(best_penalty + 0.5, ax1.get_ylim()[1] * 0.9, f'Chosen w={best_penalty} (min cost)', color='gray')
    ax1.legend(loc='upper left', fontsize=8)

    ax2 = ax1.twinx()
    ax2.plot(penalty_df['penalty'], penalty_df['RMSE'], 's--', color='steelblue', label='RMSE')
    ax2.set_ylabel('RMSE (kg) — lower is more accurate', color='steelblue')
    ax2.tick_params(axis='y', labelcolor='steelblue')

    plt.title("Operational Cost vs. Accuracy Trade-off\nPenalty chosen at the point of minimum total cost, not maximum shortfall reduction")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "penalty_cost_benefit_curve.png"), dpi=120)
    plt.close()
    penalty_df.to_csv(os.path.join(OUTPUT_DIR, "penalty_sweep_results.csv"), index=False)

    # 7b. Hyperparameter tuning (train-internal CV, no leakage) at the chosen penalty
    best_params = tune_ghost_hyperparams(train, BASE_FEATURES, best_penalty)
    print(f"\nBest LightGBM ghost params: {best_params}")

    # 7c. Compare feature sets on VALIDATION: base -> extended -> extended+friend's features
    _, _, predict_base_feats = fit_hare(train, BASE_FEATURES, best_penalty, best_params)
    train_fe = engineer_features(train)  # already engineered; kept for clarity
    _, _, predict_ghost_feats = fit_hare(train_fe, GHOST_FEATURES, best_penalty, best_params)
    _, _, predict_ghost_feats_v2 = fit_hare(train_fe, GHOST_FEATURES_V2, best_penalty, best_params)

    val_metrics_basefeat = shortfall_metrics(val['food_waste_kg'].values, predict_base_feats(val))
    val_metrics_ghostfeat = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats(val))
    val_metrics_ghostfeat_v2 = shortfall_metrics(val['food_waste_kg'].values, predict_ghost_feats_v2(val))
    print("\n[VAL] HARE (base features):", val_metrics_basefeat)
    print("[VAL] HARE (extended ghost features):", val_metrics_ghostfeat)
    print("[VAL] HARE (extended + friend's features):", val_metrics_ghostfeat_v2)

    feature_options = {
        'base': (BASE_FEATURES, val_metrics_basefeat['total_cost']),
        'extended': (GHOST_FEATURES, val_metrics_ghostfeat['total_cost']),
        'extended_v2': (GHOST_FEATURES_V2, val_metrics_ghostfeat_v2['total_cost']),
    }
    best_feature_name = min(feature_options, key=lambda k: feature_options[k][1])
    final_features = feature_options[best_feature_name][0]
    print(f"\n>>> Selected feature set (min total cost on val): {best_feature_name}")

    # 7d. Compare ghost-layer backends (from friend's code: XGBoost, CatBoost)
    # at the chosen penalty + feature set. Still asymmetrically weighted --
    # this tests which tree library learns the weighted residual best, not
    # whether the asymmetric weighting itself helps (that's settled above).
    ghost_options = {}
    for ghost_type in ['lightgbm', 'xgboost', 'catboost']:
        try:
            _, _, predict_gt = fit_hare(train, final_features, best_penalty, ghost_type=ghost_type)
            m = shortfall_metrics(val['food_waste_kg'].values, predict_gt(val))
            ghost_options[ghost_type] = m['total_cost']
            print(f"[VAL] Ghost={ghost_type}: total_cost={m['total_cost']:.2f}  "
                  f"shortfall_mass={m['shortfall_mass_kg']:.2f}  surplus_mass={m['surplus_mass_kg']:.2f}")
        except ImportError:
            print(f"[VAL] Ghost={ghost_type}: skipped (library not installed)")
    best_ghost_type = min(ghost_options, key=ghost_options.get) if ghost_options else 'lightgbm'
    print(f"\n>>> Selected ghost backend (min total cost on val): {best_ghost_type}")

    # Re-tune hyperparameters for the winning backend only if it's LightGBM
    # (tune_ghost_hyperparams is LightGBM-specific; XGBoost/CatBoost use
    # their own defaults here -- extend tune_ghost_hyperparams similarly
    # if one of them wins consistently across datasets).
    if best_ghost_type != 'lightgbm':
        best_params = None

    # ---------- FINAL MODEL: refit on train, touch test exactly once ----------
    final_base, final_ghost, final_predict = fit_hare(
        train, final_features, best_penalty, best_params, ghost_type=best_ghost_type
    )

    test_pred_hare = final_predict(test)
    test_pred_lm = lm.predict(test[BASE_FEATURES])
    test_pred_rf = rf.predict(test[BASE_FEATURES])
    test_pred_buffer = predict_buffer(test)
    test_pred_quantile = predict_quantile(test)
    actual = test['food_waste_kg'].values

    print("\n================ FINAL TEST-SET RESULTS (touched once) ================")
    print("Baseline LM:", regression_metrics(actual, test_pred_lm), shortfall_metrics(actual, test_pred_lm))
    print("Random Forest:", regression_metrics(actual, test_pred_rf), shortfall_metrics(actual, test_pred_rf))
    print(f"Safety Buffer (LM+{best_k:.2f}kg):", regression_metrics(actual, test_pred_buffer), shortfall_metrics(actual, test_pred_buffer))
    print(f"Quantile LGBM (q={best_quantile:.2f}):", regression_metrics(actual, test_pred_quantile), shortfall_metrics(actual, test_pred_quantile))
    print("HARE (final):", regression_metrics(actual, test_pred_hare), shortfall_metrics(actual, test_pred_hare))


    # ---------- Cross-validated evidence for the final model (for the paper) ----------
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmse, cv_shortfall_mass = [], []
    # Rolling-origin CV: each fold trains on an earlier chronological block and
    # evaluates on the block right after it -- matches real deployment, where
    # you only ever have the past to predict the future. Do NOT shuffle here.
    full_train_val = pd.concat([train, val], ignore_index=True).sort_values('date').reset_index(drop=True)
    for tr_idx, te_idx in tscv.split(full_train_val):
        fold_train = full_train_val.iloc[tr_idx]
        fold_eval = full_train_val.iloc[te_idx]
        _, _, fold_predict = fit_hare(fold_train, final_features, best_penalty, best_params, ghost_type=best_ghost_type)
        fold_pred = fold_predict(fold_eval)
        cv_rmse.append(np.sqrt(mean_squared_error(fold_eval['food_waste_kg'], fold_pred)))
        cv_shortfall_mass.append(shortfall_mass_scorer(fold_eval['food_waste_kg'].values, fold_pred))

    print(f"\n5-fold CV (train+val, final model): "
          f"RMSE {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}, "
          f"shortfall mass {np.mean(cv_shortfall_mass):.2f} ± {np.std(cv_shortfall_mass):.2f} kg")

    # ---------- SHAP on final ghost model ----------
    explainer = shap.TreeExplainer(final_ghost)
    shap_sample = test[final_features].iloc[:60]
    shap_values = explainer.shap_values(shap_sample)
    shap_imp = pd.DataFrame({
        'Feature': final_features,
        'Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=shap_imp, x='Importance', y='Feature', hue='Feature',
                palette='crest', legend=False)
    plt.title("Final Model — Ghost Layer Feature Attribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_shap.png"), dpi=120)
    plt.close()

    # ---------- Save final comparison table ----------
    final_table = pd.DataFrame([
        {'Model': 'Baseline LM', **regression_metrics(actual, test_pred_lm), **shortfall_metrics(actual, test_pred_lm)},
        {'Model': 'Random Forest', **regression_metrics(actual, test_pred_rf), **shortfall_metrics(actual, test_pred_rf)},
        {'Model': f'Safety Buffer (LM+{best_k:.2f}kg)', **regression_metrics(actual, test_pred_buffer), **shortfall_metrics(actual, test_pred_buffer)},
        {'Model': f'Quantile LGBM (q={best_quantile:.2f})', **regression_metrics(actual, test_pred_quantile), **shortfall_metrics(actual, test_pred_quantile)},
        {'Model': 'HARE (final)', **regression_metrics(actual, test_pred_hare), **shortfall_metrics(actual, test_pred_hare)},
    ])
    final_table.to_csv(os.path.join(OUTPUT_DIR, "final_test_results.csv"), index=False)
    print(f"\nSaved final results to {OUTPUT_DIR}/final_test_results.csv")
    print(final_table.to_string(index=False))

    # ---------- Cost-ratio sensitivity analysis (operational domain) ----------
    # Includes the two control baselines so the sensitivity plot answers
    # "does HARE still win once compared against the obvious simpler fixes?"
    # not just against LM/RF.
    all_shortfall_masses = {
        'Baseline LM': shortfall_metrics(actual, test_pred_lm)['shortfall_mass_kg'],
        'Random Forest': shortfall_metrics(actual, test_pred_rf)['shortfall_mass_kg'],
        f'Safety Buffer (+{best_k:.1f}kg)': shortfall_metrics(actual, test_pred_buffer)['shortfall_mass_kg'],
        f'Quantile LGBM (q={best_quantile:.2f})': shortfall_metrics(actual, test_pred_quantile)['shortfall_mass_kg'],
        'HARE (final)': shortfall_metrics(actual, test_pred_hare)['shortfall_mass_kg'],
    }
    all_surplus_masses = {
        'Baseline LM': shortfall_metrics(actual, test_pred_lm)['surplus_mass_kg'],
        'Random Forest': shortfall_metrics(actual, test_pred_rf)['surplus_mass_kg'],
        f'Safety Buffer (+{best_k:.1f}kg)': shortfall_metrics(actual, test_pred_buffer)['surplus_mass_kg'],
        f'Quantile LGBM (q={best_quantile:.2f})': shortfall_metrics(actual, test_pred_quantile)['surplus_mass_kg'],
        'HARE (final)': shortfall_metrics(actual, test_pred_hare)['surplus_mass_kg'],
    }
    cost_ratio_sensitivity(
        shortfall_masses=all_shortfall_masses,
        surplus_masses=all_surplus_masses,
        reference_model='HARE (final)',
    )



if __name__ == "__main__":
    main()


Train: 547  Val: 182  Test: 182
After outlier trimming -> Train: 527  Val: 173  Test: 177

[VAL] Baseline LM: {'RMSE': np.float64(5.1293755802527485), 'MAE': 4.24598342992096, 'R2': 0.8052973781304028}
[VAL] Random Forest: {'RMSE': np.float64(5.813091839437628), 'MAE': 4.723485034509184, 'R2': 0.749932548223554}

[VAL] Penalty sweep (w = 1..50):
 penalty  critical_shortfalls  shortfall_mass_kg  max_shortfall_kg   bias_kg     RMSE       R2
       1                   27         307.524202         12.411350 -0.883183 5.436126 0.781314
       2                   24         263.005296         10.795804 -1.461071 5.511387 0.775216
       3                   22         260.737032         10.030699 -1.486449 5.465196 0.778968
       5                   17         227.056505         10.469757 -2.031721 5.702897 0.759323
       7                   14         191.136838          9.382837 -2.373201 5.704174 0.759216
      10                   11         170.119535          9.258656 -2.795925 5.913

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calculate_metrics(y_true, y_pred, penalty_ratio=3.0):
    """
    Calculates operational metrics including shortfall mass, surplus mass, and total cost.
    """
    errors = y_true - y_pred

    # Shortfall: Actual > Predicted (Positive errors)
    shortfall_mass = np.sum(np.maximum(errors, 0))
    critical_shortfalls = np.sum(errors > 0)

    # Surplus: Predicted > Actual (Negative errors)
    surplus_mass = np.sum(np.maximum(-errors, 0))

    # Total Cost: Shortfall weighted by penalty_ratio, surplus weighted by 1.0
    total_cost = (shortfall_mass * penalty_ratio) + surplus_mass

    return {
        'critical_shortfalls': critical_shortfalls,
        'shortfall_mass': shortfall_mass,
        'surplus_mass': surplus_mass,
        'total_cost': total_cost
    }

def plot_cost_sensitivity(y_true, models_dict, penalty_ratios=np.linspace(1, 10, 100)):
    """
    Plots the Total Cost across different Shortfall:Surplus penalty ratios to find breakeven points.

    models_dict: dict mapping 'Model Name' to its y_pred array
    """
    plt.figure(figsize=(10, 6))

    for model_name, y_pred in models_dict.items():
        costs = []
        for r in penalty_ratios:
            metrics = calculate_metrics(y_true, y_pred, penalty_ratio=r)
            costs.append(metrics['total_cost'])

        plt.plot(penalty_ratios, costs, label=model_name, linewidth=2)

    plt.title('Total Operational Cost vs. Shortfall Penalty Ratio')
    plt.xlabel('Penalty Ratio (Shortfall Cost / Surplus Cost)')
    plt.ylabel('Total Operational Cost')
    plt.axvline(x=2.614, color='gray', linestyle=':', label='HARE vs Baseline LM Breakeven')
    plt.axvline(x=5.091, color='black', linestyle=':', label='HARE vs Safety Buffer Breakeven')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def translate_to_esg_impact(shortfall_mass_kg, surplus_mass_kg,
                             co2_factor_per_kg=2.5,  # kg CO2e per kg food waste
                             water_factor_per_kg=1000): # Liters of water per kg
    """
    Translates operational surplus (waste) and shortfall into environmental impacts.

    Parameters:
    - co2_factor_per_kg: Global average emission factor (~2.5 kg CO2e / kg food waste, FAO)
    - water_factor_per_kg: Virtual water footprint (~1,000 L / kg composite food)
    """
    # Waste impact from over-prediction (direct landfill/incineration & embedded emissions)
    waste_co2_kg = surplus_mass_kg * co2_factor_per_kg
    embedded_water_m3 = (surplus_mass_kg * water_factor_per_kg) / 1000.0  # m³

    # Unmet service impact from under-prediction
    shortfall_co2_kg = shortfall_mass_kg * co2_factor_per_kg

    return {
        'waste_co2_kg': round(waste_co2_kg, 2),
        'embedded_water_m3': round(embedded_water_m3, 2),
        'shortfall_co2_kg': round(shortfall_co2_kg, 2)
    }

# Example usage for HARE vs. Baseline LM
hare_esg = translate_to_esg_impact(shortfall_mass_kg=90.48, surplus_mass_kg=896.82)
baseline_esg = translate_to_esg_impact(shortfall_mass_kg=272.74, surplus_mass_kg=420.43)

print("HARE ESG Profile:", hare_esg)
print("Baseline LM ESG Profile:", baseline_esg)

HARE ESG Profile: {'waste_co2_kg': 2242.05, 'embedded_water_m3': 896.82, 'shortfall_co2_kg': 226.2}
Baseline LM ESG Profile: {'waste_co2_kg': 1051.08, 'embedded_water_m3': 420.43, 'shortfall_co2_kg': 681.85}


In [ ]:
import numpy as np
from scipy import stats
from sklearn.model_selection import KFold  # Or TimeSeriesSplit for temporal data


def evaluate_10fold_significance(
    model_a_preds_by_fold, model_b_preds_by_fold, y_true_by_fold, loss_fn
):
    """Computes fold-level summary loss and runs Wilcoxon and DM tests across 10 folds.

    Parameters:
    -----------
    model_a_preds_by_fold : list of np.ndarray
        Predictions per fold for Model A (HARE).
    model_b_preds_by_fold : list of np.ndarray
        Predictions per fold for Model B (Baseline).
    y_true_by_fold : list of np.ndarray
        True target values per fold.
    loss_fn : callable
        Custom loss metric function L(y_true, y_pred).

    Returns:
    --------
    results : dict
        Contains fold loss arrays, Wilcoxon stat & p-val, and DM stat & p-val.
    """
    n_folds = len(y_true_by_fold)
    assert n_folds >= 10, "Minimum 10 folds required for Wilcoxon p < 0.05!"

    fold_losses_a = []
    fold_losses_b = []

    # Calculate mean loss per fold
    for i in range(n_folds):
        y_t = y_true_by_fold[i]
        loss_a = np.mean(loss_fn(y_t, model_a_preds_by_fold[i]))
        loss_b = np.mean(loss_fn(y_t, model_b_preds_by_fold[i]))

        fold_losses_a.append(loss_a)
        fold_losses_b.append(loss_b)

    # 1. Wilcoxon Signed-Rank Test on Fold-Level Losses
    w_stat, w_p = stats.wilcoxon(
        fold_losses_a, fold_losses_b, alternative="two-sided"
    )

    # 2. Concatenate Out-of-Sample Predictions for Diebold-Mariano Test
    y_true_all = np.concatenate(y_true_by_fold)
    y_pred_a_all = np.concatenate(model_a_preds_by_fold)
    y_pred_b_all = np.concatenate(model_b_preds_by_fold)

    dm_stat, dm_p = diebold_mariano_test(
        y_true_all, y_pred_a_all, y_pred_b_all, h=1, loss_fn=loss_fn
    )

    return {
        "fold_losses_hare": fold_losses_a,
        "fold_losses_base": fold_losses_b,
        "wilcoxon_stat": w_stat,
        "wilcoxon_p_value": w_p,
        "dm_stat": dm_stat,
        "dm_p_value": dm_p,
    }


def diebold_mariano_test(y_true, y_pred1, y_pred2, h=1, loss_fn=None):
    """Diebold-Mariano test with Harvey et al.

    (1997) small-sample correction.
    """
    if loss_fn is None:
        loss_fn = lambda y, y_hat: (y - y_hat) ** 2

    d = loss_fn(y_true, y_pred1) - loss_fn(y_true, y_pred2)
    T = len(d)
    mean_d = np.mean(d)

    def autocovariance(x, lag):
        n = len(x)
        x_mean = np.mean(x)
        return (
            np.sum((x[: n - lag] - x_mean) * (x[lag:] - x_mean)) / T
            if lag < n
            else 0.0
        )

    gamma_0 = autocovariance(d, 0)
    gamma_k = sum((1 - k / h) * autocovariance(d, k) for k in range(1, h))
    var_d = (gamma_0 + 2 * gamma_k) / T

    if var_d <= 0:
        return 0.0, 1.0

    dm_stat = mean_d / np.sqrt(var_d)
    harvey_correction = np.sqrt((T + 1 - 2 * h + (h / T) * (h - 1)) / T)
    dm_stat_corrected = dm_stat * harvey_correction
    p_value = 2 * (1 - stats.t.cdf(np.abs(dm_stat_corrected), df=T - 1))

    return dm_stat_corrected, p_value


# ==========================================
# Example Simulation over 10 CV Folds
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)

    # Define Asymmetric Shortfall Loss function
    def asymmetric_loss(y, y_hat):
        err = y - y_hat
        return np.where(err > 0, 3.0 * err, 1.0 * np.abs(err))

    # Generate synthetic predictions across 10 folds
    y_true_folds, hare_preds_folds, base_preds_folds = [], [], []

    for fold in range(10):
        y_true = np.random.uniform(50, 100, size=50)
        y_hare = y_true + np.random.normal(0, 2.5, size=50)  # Lower error
        y_base = y_true + np.random.normal(-1.5, 4.5, size=50)  # Higher error

        y_true_folds.append(y_true)
        hare_preds_folds.append(y_hare)
        base_preds_folds.append(y_base)

    results = evaluate_10fold_significance(
        hare_preds_folds, base_preds_folds, y_true_folds, asymmetric_loss
    )

    print("--- 10-Fold Significance Test Results ---")
    print(f"Wilcoxon Fold Stat: {results['wilcoxon_stat']:.4f}")
    print(f"Wilcoxon p-value:   {results['wilcoxon_p_value']:.4f}")
    print(f"Diebold-Mariano Stat: {results['dm_stat']:.4f}")
    print(f"DM Test p-value:      {results['dm_p_value']:.4f}")

--- 10-Fold Significance Test Results ---
Wilcoxon Fold Stat: 0.0000
Wilcoxon p-value:   0.0020
Diebold-Mariano Stat: -13.2549
DM Test p-value:      0.0000


In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.model_selection import KFold

# ==========================================
# 1. Asymmetric Metric Definitions
# ==========================================


def evaluate_metrics(y_true, y_pred, cost_shortfall=3.0, cost_surplus=1.0):
    """Calculates RMSE, Shortfall Mass (kg), Surplus Mass (kg), and Total Cost ($)."""
    errors = y_true - y_pred
    shortfall = np.sum(np.maximum(0, errors))
    surplus = np.sum(np.maximum(0, -errors))
    rmse = np.sqrt(np.mean(errors**2))
    total_cost = (shortfall * cost_shortfall) + (surplus * cost_surplus)

    return {
        "RMSE": rmse,
        "Shortfall (kg)": shortfall,
        "Surplus (kg)": surplus,
        "Total Cost ($)": total_cost,
    }


from sklearn.ensemble import GradientBoostingRegressor

class HAREPipeline:
    def __init__(self, gamma=3.0, n_estimators=100):
        self.gamma = gamma
        # Optimal cost-minimizing quantile target: alpha = gamma / (gamma + 1)
        self.alpha = gamma / (gamma + 1.0)
        self.base_model = LinearRegression()
        # Ghost layer directly optimizes asymmetric quantile pinball loss
        self.ghost_model = GradientBoostingRegressor(
            loss="quantile",
            alpha=self.alpha,
            n_estimators=n_estimators,
            random_state=42
        )

    def fit(self, X, y):
        # Stage 1: Fit Linear Base Model for macro trend
        self.base_model.fit(X, y)
        base_preds = self.base_model.predict(X)

        # Stage 2: Fit Asymmetric Ghost Layer on Residuals
        residuals = y - base_preds
        self.ghost_model.fit(X, residuals)
        return self

    def predict(self, X):
        base_preds = self.base_model.predict(X)
        ghost_preds = self.ghost_model.predict(X)
        return base_preds + ghost_preds


# ==========================================
# 3. Benchmark Models Setup
# ==========================================


def get_benchmark_models(gamma=3.0):
    return {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=100, random_state=42
        ),
        "Quantile Reg (q=0.75)": QuantileRegressor(
            quantile=0.75, alpha=0.0, solver="highs"
        ),
        "LightGBM": LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
        "HARE (Full Pipeline)": HAREPipeline(gamma=gamma),
    }


# ==========================================
# 4. Multi-Domain Synthetic Data Generator
# ==========================================


def generate_domain_dataset(domain_type="university", n_samples=500):
    np.random.seed(42)
    X = np.random.randn(n_samples, 5)

    # Feature-dependent variance (Heteroscedasticity)
    # Higher values of feat_0 create higher demand uncertainty
    hetero_noise = (1.0 + 1.5 * np.abs(X[:, 0])) * np.random.normal(0, 8, n_samples)

    if domain_type == "university":
        y = 200 + 15 * X[:, 0] + 10 * X[:, 1] + hetero_noise
    elif domain_type == "hospital":
        y = 80 + 5 * X[:, 0] + 3 * X[:, 1] + (0.5 * hetero_noise)
    elif domain_type == "hotel":
        spikes = np.random.choice([0, 35], size=n_samples, p=[0.85, 0.15])
        y = 150 + 20 * X[:, 0] + 8 * X[:, 1] + spikes + hetero_noise

    return pd.DataFrame(X, columns=[f"feat_{i}" for i in range(5)]), pd.Series(y)


# ==========================================
# 5. Execution: Multi-Domain & Benchmarks
# ==========================================
domains = ["university", "hospital", "hotel"]
domain_results = []

for domain in domains:
    X, y = generate_domain_dataset(domain_type=domain)
    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    models = get_benchmark_models(gamma=3.0)

    for name, model in models.items():
        y_trues, y_preds = [], []

        for train_idx, test_idx in kf.split(X):
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

            model.fit(X_tr, y_tr)
            preds = model.predict(X_te)

            y_trues.extend(y_te)
            y_preds.extend(preds)

        m = evaluate_metrics(np.array(y_trues), np.array(y_preds))
        m["Domain"] = domain.capitalize()
        m["Model"] = name
        domain_results.append(m)

df_results = pd.DataFrame(domain_results)
# Add this at the end of Section 5
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("\n================ MULTI-DOMAIN BENCHMARK RESULTS ================")
print(df_results.to_string(index=False))


================ MULTI-DOMAIN BENCHMARK RESULTS ================
     RMSE  Shortfall (kg)  Surplus (kg)  Total Cost ($)     Domain                 Model
19.219163     3546.551665   3568.735430    14208.390424 University     Linear Regression
20.367707     3713.241083   3744.232419    14883.955667 University         Random Forest
21.671604     1702.118167   6721.501796    11827.856296 University Quantile Reg (q=0.75)
21.214876     3975.263139   3996.862967    15922.652386 University              LightGBM
22.077263     1637.302304   6442.499031    11354.405942 University  HARE (Full Pipeline)
 9.609582     1773.275832   1784.367715     7104.195212   Hospital     Linear Regression
10.071185     1852.004229   1861.296601     7417.309288   Hospital         Random Forest
10.835802      851.059083   3360.750898     5913.928148   Hospital Quantile Reg (q=0.75)
10.534194     2022.702364   1975.312791     8043.419883   Hospital              LightGBM
11.038632      818.651152   3221.249516     

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, RegressorMixin
from scipy.stats import uniform, randint

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import shap

warnings.filterwarnings('ignore')

# Style configuration for publication-ready journal figures
plt.style.use('seaborn-v0_8-paper' if 'seaborn-v0_8-paper' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14
})

SEED = 123
np.random.seed(SEED)

OUTPUT_DIR = "journal_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHORTFALL_THRESHOLD = 5.0      # kg under-prediction considered "critical"
COST_PER_KG_SHORTFALL = 3.0    # Operational penalty per kg short
COST_PER_KG_SURPLUS = 1.0      # Ingredient cost per kg waste

# ============================================================
# 1. DATA LOAD & CHRONOLOGICAL SPLIT
# ============================================================
def load_and_clean(path="train.csv"):
    data = pd.read_csv(path)
    def clean_num(s):
        return pd.to_numeric(s.astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')

    num_cols = ['meals_served', 'kitchen_staff', 'temperature_C', 'humidity_percent', 'past_waste_kg', 'food_waste_kg']
    for col in num_cols:
        data[col] = clean_num(data[col])

    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month
    data['day_of_week'] = data['date'].dt.dayofweek
    data = data.drop(columns=['ID'], errors='ignore')
    return data.sort_values('date').reset_index(drop=True)

def three_way_split(data, val_size=0.2, test_size=0.2):
    n = len(data)
    n_test = int(n * test_size)
    n_val = int(n * val_size)
    n_train = n - n_val - n_test
    return (data.iloc[:n_train].reset_index(drop=True),
            data.iloc[n_train:n_train + n_val].reset_index(drop=True),
            data.iloc[n_train + n_val:].reset_index(drop=True))

def fit_apply_outliers(train, val, test, cols):
    bounds = {}
    for c in cols:
        q1, q3 = train[c].quantile(0.25), train[c].quantile(0.75)
        iqr = q3 - q1
        bounds[c] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

    def trim(df):
        mask = pd.Series(True, index=df.index)
        for c, (lo, hi) in bounds.items():
            mask &= df[c].between(lo, hi)
        return df[mask].reset_index(drop=True)

    return trim(train), trim(val), trim(test)

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
BASE_FEATURES = [
    "meals_served", "kitchen_staff", "past_waste_kg", "special_event",
    "temperature_C", "humidity_percent", "day_of_week", "day", "month"
]

def engineer_features(df):
    df = df.copy()
    df['staff_efficiency'] = df['meals_served'] / (df['kitchen_staff'] + 1e-5)
    df['demand_pressure'] = df['meals_served'] / (df['kitchen_staff'] + 1)
    df['env_score'] = df['temperature_C'] * df['humidity_percent']
    df['event_risk'] = df['special_event'] * df['meals_served']
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    # Non-linear & Interaction features
    df['meals_x_humidity'] = df['meals_served'] * df['humidity_percent']
    df['past_waste_x_temp'] = df['past_waste_kg'] * df['temperature_C']
    df['meals_served_sq'] = df['meals_served'] ** 2
    df['temperature_C_sq'] = df['temperature_C'] ** 2
    return df

ALL_FEATURES = BASE_FEATURES + [
    'staff_efficiency', 'demand_pressure', 'env_score', 'event_risk',
    'is_weekend', 'meals_x_humidity', 'past_waste_x_temp',
    'meals_served_sq', 'temperature_C_sq'
]

# ============================================================
# 3. HARE ESTIMATOR (ASYMMETRIC RESIDUAL MODEL)
# ============================================================
class HAREEstimator(BaseEstimator, RegressorMixin):
    def __init__(self, penalty=20.0, ghost_type='lightgbm', n_estimators=300, learning_rate=0.03):
        self.penalty = penalty
        self.ghost_type = ghost_type
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate

    def fit(self, X, y):
        X_df = pd.DataFrame(X)
        y_arr = np.ravel(y)
        self.base_ = LinearRegression().fit(X_df, y_arr)
        resid = y_arr - self.base_.predict(X_df)
        weights = np.where(resid > 0, self.penalty, 1.0)

        if self.ghost_type == 'lightgbm':
            self.ghost_ = lgb.LGBMRegressor(
                n_estimators=self.n_estimators, learning_rate=self.learning_rate,
                random_state=SEED, verbose=-1, n_jobs=1
            )
        elif self.ghost_type == 'xgboost':
            self.ghost_ = xgb.XGBRegressor(
                n_estimators=self.n_estimators, learning_rate=self.learning_rate,
                random_state=SEED, n_jobs=1
            )
        elif self.ghost_type == 'catboost':
            self.ghost_ = cb.CatBoostRegressor(
                iterations=self.n_estimators, learning_rate=self.learning_rate,
                random_seed=SEED, verbose=False, thread_count=1
            )
        self.ghost_.fit(X_df, resid, sample_weight=weights)
        return self

    def predict(self, X):
        X_df = pd.DataFrame(X)
        return self.base_.predict(X_df) + self.ghost_.predict(X_df)

# ============================================================
# 4. EVALUATION METRICS
# ============================================================
def compute_metrics(actual, pred):
    resid = actual - pred
    shortfalls = resid[resid > 0]
    surpluses = -resid[resid < 0]

    shortfall_mass = np.sum(shortfalls)
    surplus_mass = np.sum(surpluses)
    total_cost = (shortfall_mass * COST_PER_KG_SHORTFALL) + (surplus_mass * COST_PER_KG_SURPLUS)

    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'R2': r2_score(actual, pred),
        'Critical_Shortfalls': int(np.sum(resid > SHORTFALL_THRESHOLD)),
        'Shortfall_Mass_kg': shortfall_mass,
        'Operational_Cost_$': total_cost,
        'Mean_Bias_kg': np.mean(resid)
    }

# ============================================================
# 5. MAIN BENCHMARK & PLOTTING PIPELINE
# ============================================================
def main():
    data = load_and_clean("train.csv")
    train, val, test = three_way_split(data)

    outlier_cols = ["meals_served", "food_waste_kg", "temperature_C", "past_waste_kg"]
    train, val, test = fit_apply_outliers(train, val, test, outlier_cols)

    train = engineer_features(train)
    val = engineer_features(val)
    test = engineer_features(test)

    X_tr, y_tr = train[ALL_FEATURES], train['food_waste_kg'].values
    X_val, y_val = val[ALL_FEATURES], val['food_waste_kg'].values
    X_te, y_te = test[ALL_FEATURES], test['food_waste_kg'].values

    # --- PENALTY SWEEP ON VALIDATION SET ---
    penalties = [1, 2, 5, 10, 15, 20, 25, 30, 40, 50]
    sweep_records = []
    for p in penalties:
        m = HAREEstimator(penalty=p, ghost_type='lightgbm').fit(X_tr, y_tr)
        preds = m.predict(X_val)
        res = compute_metrics(y_val, preds)
        sweep_records.append({'Penalty': p, **res})

    sweep_df = pd.DataFrame(sweep_records)
    best_penalty = int(sweep_df.loc[sweep_df['Operational_Cost_$'].idxmin(), 'Penalty'])

    # --- MODEL BENCHMARK SUITE ---
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(alpha=1.0),
        'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=SEED),
        'Random Forest': RandomForestRegressor(n_estimators=300, random_state=SEED),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, random_state=SEED),
        'LightGBM (Standard)': lgb.LGBMRegressor(n_estimators=300, random_state=SEED, verbose=-1),
        'XGBoost (Standard)': xgb.XGBRegressor(n_estimators=300, random_state=SEED),
        'CatBoost (Standard)': cb.CatBoostRegressor(iterations=300, random_seed=SEED, verbose=False),
        'HARE-LightGBM (Ours)': HAREEstimator(penalty=best_penalty, ghost_type='lightgbm'),
        'HARE-XGBoost (Ours)': HAREEstimator(penalty=best_penalty, ghost_type='xgboost'),
        'HARE-CatBoost (Ours)': HAREEstimator(penalty=best_penalty, ghost_type='catboost')
    }

    test_results = []
    predictions_dict = {}

    for name, model in models.items():
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        predictions_dict[name] = preds
        metrics = compute_metrics(y_te, preds)
        test_results.append({'Model': name, **metrics})

    results_df = pd.DataFrame(test_results).sort_values('Operational_Cost_$')
    results_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_results.csv"), index=False)

    # Save LaTeX Table for Paper
    with open(os.path.join(OUTPUT_DIR, "table_comparison.tex"), "w") as f:
        f.write(results_df.to_latex(index=False, float_format="%.3f"))

    # ============================================================
    # 6. JOURNAL FIGURES GENERATION
    # ============================================================

    # FIGURE 1: Multi-Model Performance Comparison (Bar Charts)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=results_df, x='RMSE', y='Model', ax=axes[0], palette='Blues_r')
    axes[0].set_title("(a) Predictive Error (RMSE in kg)")
    axes[0].set_xlabel("RMSE (Lower is Better)")

    sns.barplot(data=results_df, x='Shortfall_Mass_kg', y='Model', ax=axes[1], palette='Reds_r')
    axes[1].set_title("(b) Operational Shortfall Mass (kg)")
    axes[1].set_xlabel("Shortfall Mass (Lower is Safer)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig1_Model_Comparison.png"), dpi=300)
    plt.close()

    # FIGURE 2: Cost-Benefit Curve (Penalty Sweep)
    fig, ax1 = plt.subplots(figsize=(8, 5))
    color = 'tab:red'
    ax1.set_xlabel('Under-prediction Penalty Weight ($w$)')
    ax1.set_ylabel('Shortfall Mass (kg)', color=color)
    ax1.plot(sweep_df['Penalty'], sweep_df['Shortfall_Mass_kg'], 'o-', color=color, linewidth=2)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Operational Cost ($)', color=color)
    ax2.plot(sweep_df['Penalty'], sweep_df['Operational_Cost_$'], 's--', color=color, linewidth=2)
    ax2.tick_params(axis='y', labelcolor=color)

    plt.axvline(x=best_penalty, color='gray', linestyle=':', label=f'Optimal w={best_penalty}')
    plt.title("Operational Safety vs. Economic Cost Trade-off")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig2_Cost_Benefit_Tradeoff.png"), dpi=300)
    plt.close()

    # FIGURE 3: Actual vs Predicted Scatter (Selected Models)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    selected_models = ['Linear Regression', 'Random Forest', 'HARE-LightGBM (Ours)']

    for idx, m_name in enumerate(selected_models):
        preds = predictions_dict[m_name]
        axes[idx].scatter(y_te, preds, alpha=0.6, edgecolors='k', linewidth=0.5, color='teal')
        axes[idx].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=2)
        axes[idx].set_title(m_name)
        axes[idx].set_xlabel("Actual Waste (kg)")
        if idx == 0:
            axes[idx].set_ylabel("Predicted Waste (kg)")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig3_Predicted_vs_Actual.png"), dpi=300)
    plt.close()

    # FIGURE 4: Residual Density Distribution (Error Asymmetry)
    plt.figure(figsize=(9, 5))
    for m_name in ['Linear Regression', 'Random Forest', 'HARE-LightGBM (Ours)']:
        resids = y_te - predictions_dict[m_name]
        sns.kdeplot(resids, label=m_name, fill=True, alpha=0.25)

    plt.axvline(0, color='black', linestyle='--', label='Zero Error')
    plt.axvline(SHORTFALL_THRESHOLD, color='red', linestyle=':', label='Critical Shortfall Limit')
    plt.title("Residual Density Distribution ($y_{actual} - y_{pred}$)")
    plt.xlabel("Residual Error (kg) — Positive represents under-prediction")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig4_Residual_Density.png"), dpi=300)
    plt.close()

    # FIGURE 5: SHAP Attribution (HARE Ghost Layer)
    hare_ghost = models['HARE-LightGBM (Ours)'].ghost_
    explainer = shap.TreeExplainer(hare_ghost)
    shap_vals = explainer.shap_values(X_te)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_te, show=False)
    plt.title("HARE Ghost Layer — Feature Attribution (SHAP)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig5_SHAP_Feature_Importance.png"), dpi=300)
    plt.close()

    print("\n================ BENCHMARK COMPLETED SUCCESSFULLY ================")
    print(results_df.to_string(index=False))
    print(f"\nOutputs and High-Res Figures saved in directory: '{OUTPUT_DIR}/'")

if __name__ == "__main__":
    main()


================ BENCHMARK COMPLETED SUCCESSFULLY ================
               Model     RMSE      MAE       R2  Critical_Shortfalls  Shortfall_Mass_kg  Operational_Cost_$  Mean_Bias_kg
HARE-LightGBM (Ours) 6.549029 5.380395 0.648622                    6         101.200543         1154.731003     -4.236886
HARE-CatBoost (Ours) 6.702441 5.640041 0.631967                    6          86.909390         1172.106089     -4.658014
 HARE-XGBoost (Ours) 5.805500 4.694205 0.723879                   12         188.424614         1207.723560     -2.565114
   Linear Regression 4.866205 3.885488 0.806000                   17         264.205626         1216.142679     -0.900114
    Ridge Regression 4.873386 3.878967 0.805427                   17         267.935964         1222.449110     -0.851442
 CatBoost (Standard) 5.424167 4.463775 0.758961                   25         301.545705         1393.179531     -1.056479
   Gradient Boosting 5.678054 4.607099 0.735869                   28         3